# Legal Retrieval Experiments (H1–H11)

## How to run

1. Keep this notebook next to `configs/` and `data/` (both ship in this folder).
2. Install the dependencies (Section 0). H1, H6–H11 (BM25) only need `numpy` and `PyYAML`. H2–H5 (dense / hybrid / rerank) also need `sentence-transformers` and `faiss-cpu`, and download Hugging Face models (`BAAI/bge-m3`, `BAAI/bge-reranker-v2-m3`).
3. Edit the configuration cell: `PROJECT_ROOT`, `BENCHMARK_PATH`, `RUN_DENSE`.
4. Run all cells from top to bottom.

Outputs:
- `artifacts/runs/<hypothesis-id>/<run-id>/` — one folder per run: `config.yaml`, `summary.json`, `per_query.jsonl`
- `artifacts/reports/summary.csv` — one row per hypothesis (latest run)
- `artifacts/reports/vs_baseline.csv` — each hypothesis minus its baseline

Hypotheses: each config changes only the variables listed in `hypothesis.variables` relative to the config it `extends` (its baseline).

| ID | Baseline | Change |
|---|---|---|
| H1 | — | BM25 + Article/Clause/Point units |
| H2 | H1 | Dense only (bge-m3) |
| H3 | H1 | Hybrid BM25 + dense, RRF |
| H4 | H3 | + cross-encoder reranking |
| H5 | H3 | Fixed-length 220-word chunks (hybrid) |
| H6 | H1 | Fixed-length 220-word chunks (BM25) |
| H7 | H1 | Clause/Point children only |
| H8 | H7 | + Article heading prepended to children |
| H9 | H8 | Article first, then children inside it |
| H10 | H1 | + temporal pre-filtering |
| H11 | H9 | + temporal pre-filtering |

Metrics are reported twice: strict (`recall@10`, exact legal unit) and Article-level (`article.recall@10`, any hit inside the gold unit's Article version).

## 0. Setup & configuration

Set `RUN_DENSE = True` to also run H2–H5 (requires the extra packages, enough RAM, and a GPU is recommended).

In [1]:
# Uncomment to install dependencies
# !pip install -q numpy PyYAML pandas
# !pip install -q "sentence-transformers>=3.0" "faiss-cpu>=1.8.0"   # only needed for H2-H5

In [2]:
from pathlib import Path

# Repository root (contains configs/ and data/). Keep "." when the notebook is in
# the repo root; otherwise set an absolute path.
PROJECT_ROOT = Path(".").resolve()

# Keep downloaded Hugging Face models inside the project instead of ~/.cache.
# Must be set before sentence-transformers is imported.
import os
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".hf_cache"))
# Reduces CUDA memory fragmentation on small GPUs (must be set before torch uses CUDA).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Experiment outputs.
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RUNS_DIR = ARTIFACTS_DIR / "runs"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

# Query set used by every hypothesis (.jsonl / .json / .csv). Replace with a
# dev/test set when available.
BENCHMARK_PATH = "data/benchmarks/schema_example.jsonl"

# True: also run H2-H5 (downloads bge-m3 and bge-reranker-v2-m3).
RUN_DENSE = False

LOG_LEVEL = "INFO"

assert (PROJECT_ROOT / "configs" / "base.yaml").is_file(), f"configs/base.yaml not found in {PROJECT_ROOT}"
assert (PROJECT_ROOT / "data" / "enriched").is_dir(), f"data/enriched not found in {PROJECT_ROOT}"
print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /home/barokhuu/workspace/URA/notebook


In [3]:
import csv
import glob
import hashlib
import json
import logging
import math
import platform
import random
import re
import subprocess
import sys
import time
import traceback
import unicodedata
import uuid
from abc import ABC, abstractmethod
from collections import Counter, defaultdict
from copy import deepcopy
from dataclasses import dataclass, field, asdict
from datetime import date, datetime, timezone
from importlib import metadata
from typing import Any, Iterable

import numpy as np
import yaml


def configure_logging(level: str = "INFO") -> None:          # logging_utils.py
    logging.basicConfig(
        level=getattr(logging, level.upper(), logging.INFO),
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        force=True,
    )


configure_logging(LOG_LEVEL)
logger = logging.getLogger("legal_retrieval")

## 1. Core modules

Run these cells once; they only define functions and classes.

### 1.1 Paths — `paths.py`

In [4]:
def project_root() -> Path:
    return PROJECT_ROOT


def resolve_project_path(value: str | Path | None) -> Path | None:
    if value is None:
        return None
    p = Path(value).expanduser()
    return p if p.is_absolute() else project_root() / p

### 1.2 Text normalization — `text.py`

In [5]:
_WS = re.compile(r"\s+")
_TOKEN = re.compile(r"\w+", re.UNICODE)


def normalize_whitespace(text: str) -> str:
    return _WS.sub(" ", text or "").strip()


def fold_vietnamese(text: str) -> str:
    text = unicodedata.normalize("NFD", text or "")
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("đ", "d").replace("Đ", "D")


def sparse_normalize(text: str, lowercase: bool = True, fold: bool = True) -> str:
    out = normalize_whitespace(text)
    if fold:
        out = fold_vietnamese(out)
    if lowercase:
        out = out.lower()
    return out


def tokenize(text: str) -> list[str]:
    return _TOKEN.findall(text.lower())

### 1.3 Data models — `models.py` + `io_utils.py`

In [6]:
@dataclass
class RetrievalRecord:
    record_id: str
    text: str
    sparse_text: str
    source_unit_ids: list[str]
    metadata: dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "RetrievalRecord":
        return cls(**data)


@dataclass
class SearchHit:
    record_id: str
    score: float
    rank: int = 0
    source: str = ""
    record: RetrievalRecord | None = None
    component_scores: dict[str, float] = field(default_factory=dict)

    def to_dict(self, include_text: bool = True) -> dict[str, Any]:
        payload: dict[str, Any] = {
            "record_id": self.record_id,
            "rank": self.rank,
            "score": float(self.score),
            "source": self.source,
            "component_scores": {k: float(v) for k, v in self.component_scores.items()},
        }
        if self.record:
            payload["unit_id"] = self.record.record_id
            payload["source_unit_ids"] = self.record.source_unit_ids
            payload["metadata"] = self.record.metadata
            if include_text:
                payload["text"] = self.record.text
        return payload


def write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

### 1.4 Temporal parsing & filtering — `temporal.py`

In [7]:
_FULL_DATE = re.compile(r"(?<!\d)(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?!\d)")
_ISO_DATE = re.compile(r"(?<!\d)(\d{4})-(\d{1,2})-(\d{1,2})(?!\d)")
_YEAR = re.compile(r"(?<!\d)(19\d{2}|20\d{2}|2100)(?!\d)")
_TEMPORAL_YEAR_CUE = re.compile(
    r"\b(?:năm|nam|vào|vao|tại|tai|thời điểm|thoi diem|trước|truoc|sau|từ|tu|đến|den)\b",
    re.IGNORECASE,
)


@dataclass(frozen=True)
class TemporalConstraint:
    mode: str
    start: date | None
    end: date | None
    explicit: bool
    semantic_query: str

    def to_dict(self) -> dict[str, Any]:
        return {
            "mode": self.mode,
            "start": self.start.isoformat() if self.start else None,
            "end": self.end.isoformat() if self.end else None,
            "explicit": self.explicit,
        }


def parse_legal_date(value: Any) -> date | None:
    if value is None or value == "":
        return None
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    raw = str(value).strip()
    for fmt in ("%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(raw, fmt).date()
        except ValueError:
            pass
    raise ValueError(f"Unsupported legal date {value!r}; use DD/MM/YYYY or YYYY-MM-DD")


def _reference_date(cfg: dict[str, Any]) -> date:
    configured = cfg.get("reference_date")
    return parse_legal_date(configured) if configured else date.today()


def parse_temporal_query(query: str, cfg: dict[str, Any]) -> TemporalConstraint:
    '''Extract a conservative temporal constraint from a Vietnamese query.

    A bare four-digit year is only temporal when the query has a temporal cue,
    preventing document names such as "Luật Đất đai 2024" from automatically
    becoming an as-of query.
    '''
    lowered = query.lower()
    dates: list[tuple[date, tuple[int, int]]] = []
    for match in _FULL_DATE.finditer(query):
        day, month, year = map(int, match.groups())
        dates.append((date(year, month, day), match.span()))
    for match in _ISO_DATE.finditer(query):
        year, month, day = map(int, match.groups())
        dates.append((date(year, month, day), match.span()))

    years: list[tuple[int, tuple[int, int]]] = []
    if not dates and _TEMPORAL_YEAR_CUE.search(lowered):
        years = [(int(m.group(1)), m.span()) for m in _YEAR.finditer(query)]

    spans = [span for _, span in dates] + [span for _, span in years]
    semantic_query = query
    if cfg.get("strip_time_from_query", True):
        for start, end in sorted(spans, reverse=True):
            semantic_query = semantic_query[:start] + " " + semantic_query[end:]
        semantic_query = " ".join(semantic_query.split())

    if not dates and not years:
        ref = _reference_date(cfg)
        return TemporalConstraint("current", ref, ref, False, semantic_query)

    before = bool(re.search(r"\b(?:trước|truoc)\b", lowered))
    after = bool(re.search(r"\b(?:sau)\b", lowered))
    between = len(dates) >= 2 or len(years) >= 2

    if dates:
        values = sorted(value for value, _ in dates)
        if between:
            return TemporalConstraint("between", values[0], values[-1], True, semantic_query)
        point = values[0]
        mode = "before" if before else "after" if after else "as_of"
        return TemporalConstraint(mode, point, point, True, semantic_query)

    values = sorted(year for year, _ in years)
    if between:
        return TemporalConstraint("between", date(values[0], 1, 1), date(values[-1], 12, 31), True, semantic_query)
    year = values[0]
    if before:
        point = date(year, 1, 1)
        return TemporalConstraint("before", point, point, True, semantic_query)
    if after:
        point = date(year, 12, 31)
        return TemporalConstraint("after", point, point, True, semantic_query)
    return TemporalConstraint("as_of", date(year, 1, 1), date(year, 12, 31), True, semantic_query)


def record_matches_temporal(metadata: dict[str, Any], constraint: TemporalConstraint, missing_policy: str) -> bool:
    try:
        effective_from = parse_legal_date(metadata.get("effective_from"))
        effective_to = parse_legal_date(metadata.get("effective_to"))
    except ValueError:
        if missing_policy == "error":
            raise
        return missing_policy == "include"

    if effective_from is None:
        if missing_policy == "error":
            raise ValueError(f"Record {metadata.get('unit_id')} has no effective_from")
        return missing_policy == "include"

    start = constraint.start
    end = constraint.end
    assert start is not None and end is not None
    if constraint.mode == "before":
        return effective_from < start
    if constraint.mode == "after":
        return effective_from > end
    # current, as_of and between mean that the validity interval overlaps the
    # requested interval. For a precise date start == end.
    return effective_from <= end and (effective_to is None or effective_to >= start)

### 1.5 Config loading & hypothesis validation — `config.py`

In [8]:
# These fields are expected to differ between experiment files but do not alter
# the scientific treatment. They are therefore excluded from hypothesis diffing.
_HYPOTHESIS_DIFF_IGNORES = (
    "hypothesis",
    "indexing.output_dir",
)
_HF_COMMIT_RE = re.compile(r"^[0-9a-fA-F]{40}$")


def deep_merge(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]:
    out = deepcopy(base)
    for key, value in override.items():
        if key in out and isinstance(out[key], dict) and isinstance(value, dict):
            out[key] = deep_merge(out[key], value)
        else:
            out[key] = deepcopy(value)
    return out


def _load_yaml(path: Path, seen: set[Path]) -> dict[str, Any]:
    path = path.resolve()
    if path in seen:
        raise ValueError(f"Circular config inheritance detected at: {path}")
    seen.add(path)
    if not path.exists():
        raise FileNotFoundError(path)
    data = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
    parent = data.pop("extends", None)
    if parent:
        parent_path = (path.parent / parent).resolve()
        base = _load_yaml(parent_path, seen)
        data = deep_merge(base, data)
    seen.remove(path)
    return data


def _resolve_config_path(path: str | Path) -> Path:
    p = Path(path)
    if not p.is_absolute():
        # First respect a path relative to CWD; otherwise use project root.
        p = p if p.exists() else project_root() / p
    return p.resolve()


def _immediate_parent_path(path: Path) -> Path | None:
    raw = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
    parent = raw.get("extends")
    return (path.parent / parent).resolve() if parent else None


def _is_ignored_diff(path: str) -> bool:
    return any(path == prefix or path.startswith(prefix + ".") or path.startswith(prefix + "[")
               for prefix in _HYPOTHESIS_DIFF_IGNORES)


def _diff_paths(baseline: Any, current: Any, prefix: str = "") -> list[str]:
    '''Return concrete leaf paths whose resolved values differ.'''
    if isinstance(baseline, dict) and isinstance(current, dict):
        changes: list[str] = []
        for key in sorted(set(baseline) | set(current)):
            path = f"{prefix}.{key}" if prefix else str(key)
            if key not in baseline or key not in current:
                if not _is_ignored_diff(path):
                    changes.append(path)
                continue
            changes.extend(_diff_paths(baseline[key], current[key], path))
        return changes

    if isinstance(baseline, list) and isinstance(current, list):
        changes = []
        common = min(len(baseline), len(current))
        for i in range(common):
            path = f"{prefix}[{i}]"
            changes.extend(_diff_paths(baseline[i], current[i], path))
        for i in range(common, max(len(baseline), len(current))):
            path = f"{prefix}[{i}]"
            if not _is_ignored_diff(path):
                changes.append(path)
        return changes

    if baseline != current and prefix and not _is_ignored_diff(prefix):
        return [prefix]
    return []


def _change_is_declared(change: str, declared: str) -> bool:
    '''A declaration may name a leaf or intentionally authorize a subtree/list.'''
    return (
        change == declared
        or change.startswith(declared + ".")
        or change.startswith(declared + "[")
    )


def validate_hypothesis_changes(cfg: dict[str, Any], baseline: dict[str, Any] | None) -> dict[str, Any]:
    declared = list(cfg.get("hypothesis", {}).get("variables", []) or [])
    if not all(isinstance(x, str) and x.strip() for x in declared):
        raise ValueError("hypothesis.variables must be a list of non-empty config paths")
    declared = sorted(dict.fromkeys(x.strip() for x in declared))

    actual = sorted(_diff_paths(baseline or {}, cfg)) if baseline is not None else []
    undeclared = [c for c in actual if not any(_change_is_declared(c, d) for d in declared)]

    report = {
        "declared_changes": declared,
        "actual_changes": actual,
        "undeclared_changes": undeclared,
    }
    if undeclared:
        raise ValueError(
            "Hypothesis config changed variables outside hypothesis.variables.\n"
            f"Declared changes = {declared}\n"
            f"Actual changes = {actual}\n"
            f"Undeclared changes = {undeclared}"
        )
    return report


def load_config(path: str | Path) -> dict[str, Any]:
    p = _resolve_config_path(path)
    cfg = _load_yaml(p, set())
    validate_config(cfg)

    parent_path = _immediate_parent_path(p)
    baseline = _load_yaml(parent_path, set()) if parent_path is not None else None
    diff_report = validate_hypothesis_changes(cfg, baseline)

    cfg["_config_path"] = str(p)
    cfg["_baseline_config_path"] = str(parent_path) if parent_path is not None else None
    cfg["_hypothesis_diff"] = diff_report
    return cfg


def _require_hf_commit(path: str, revision: Any) -> None:
    if not isinstance(revision, str) or not _HF_COMMIT_RE.fullmatch(revision):
        raise ValueError(
            f"{path} must be pinned to a 40-character Hugging Face commit hash "
            "when runtime.reproducibility_mode=final"
        )


def validate_config(cfg: dict[str, Any]) -> None:
    mode = cfg.get("representation", {}).get("mode")
    if mode not in {"hierarchical_units", "fixed_length", "parent_child"}:
        raise ValueError(
            "representation.mode must be hierarchical_units, fixed_length, or parent_child, "
            f"got {mode!r}"
        )

    method = cfg.get("retrieval", {}).get("method")
    if method not in {"sparse", "dense", "hybrid"}:
        raise ValueError(f"retrieval.method must be sparse/dense/hybrid, got {method!r}")

    sparse_enabled = bool(cfg.get("indexing", {}).get("sparse", {}).get("enabled"))
    dense_cfg = cfg.get("indexing", {}).get("dense", {})
    dense_enabled = bool(dense_cfg.get("enabled"))
    sparse_backend = cfg.get("indexing", {}).get("sparse", {}).get("backend")
    if sparse_enabled and sparse_backend != "bm25":
        raise ValueError(f"Unsupported indexing.sparse.backend={sparse_backend!r}; currently only bm25 is available")
    if method in {"sparse", "hybrid"} and not sparse_enabled:
        raise ValueError(f"retrieval.method={method} requires indexing.sparse.enabled=true")
    if method in {"dense", "hybrid"} and not dense_enabled:
        raise ValueError(f"retrieval.method={method} requires indexing.dense.enabled=true")

    fusion = cfg.get("fusion", {}).get("method", "none")
    if fusion not in {"none", "rrf", "weighted"}:
        raise ValueError(f"Unsupported fusion.method={fusion!r}")
    if method == "hybrid" and fusion == "none":
        raise ValueError("Hybrid retrieval requires fusion.method=rrf or weighted")

    rerank = cfg.get("reranking", {})
    if rerank.get("enabled") and not rerank.get("models"):
        raise ValueError("reranking.enabled=true but reranking.models is empty")

    metrics = cfg.get("evaluation", {}).get("metrics", [])
    allowed = {"hit_rate", "precision", "recall", "mrr", "ndcg", "map"}
    unknown = set(metrics) - allowed
    if unknown:
        raise ValueError(f"Unsupported evaluation metrics: {sorted(unknown)}")

    def positive(path: str, value: Any) -> None:
        if not isinstance(value, int) or isinstance(value, bool) or value <= 0:
            raise ValueError(f"{path} must be a positive integer, got {value!r}")

    retrieval = cfg.get("retrieval", {})
    for key in ("sparse_top_k", "dense_top_k", "candidate_top_k"):
        positive(f"retrieval.{key}", retrieval.get(key))
    positive("ranking.final_top_k", cfg.get("ranking", {}).get("final_top_k"))
    for value in cfg.get("evaluation", {}).get("k_values", []):
        positive("evaluation.k_values[]", value)

    dedupe = cfg.get("ranking", {}).get("deduplicate_by")
    if dedupe not in {"none", "record_id", "source_unit"}:
        raise ValueError(f"Unsupported ranking.deduplicate_by={dedupe!r}")
    match_levels = {"record_id", "source_unit", "doc_id", "article"}
    match_by = cfg.get("evaluation", {}).get("match_by")
    if match_by not in match_levels:
        raise ValueError(f"Unsupported evaluation.match_by={match_by!r}")
    unknown_levels = set(cfg.get("evaluation", {}).get("also_match_by", [])) - match_levels
    if unknown_levels:
        raise ValueError(f"Unsupported evaluation.also_match_by: {sorted(unknown_levels)}")
    if cfg.get("corpus", {}).get("duplicate_unit_id_policy", "keep") not in {"keep", "dedupe", "error"}:
        raise ValueError("corpus.duplicate_unit_id_policy must be keep, dedupe, or error")

    fixed = cfg.get("representation", {}).get("fixed_length", {})
    size, overlap = fixed.get("chunk_size"), fixed.get("overlap")
    if mode == "fixed_length":
        positive("representation.fixed_length.chunk_size", size)
        if not isinstance(overlap, int) or isinstance(overlap, bool) or not 0 <= overlap < size:
            raise ValueError("representation.fixed_length.overlap must satisfy 0 <= overlap < chunk_size")

    parent_child = cfg.get("representation", {}).get("parent_child", {})
    if mode == "parent_child":
        if not parent_child.get("child_unit_types"):
            raise ValueError("representation.parent_child.child_unit_types must not be empty")
        positive(
            "representation.parent_child.context.parent_heading_max_words",
            parent_child.get("context", {}).get("parent_heading_max_words"),
        )
        if parent_child.get("orphan_policy", "error") not in {"error", "skip"}:
            raise ValueError("representation.parent_child.orphan_policy must be error or skip")

    parent_retrieval = retrieval.get("parent_child", {})
    if parent_retrieval.get("enabled"):
        if mode != "parent_child":
            raise ValueError("retrieval.parent_child.enabled requires representation.mode=parent_child")
        if not parent_child.get("include_parent_records"):
            raise ValueError(
                "retrieval.parent_child.enabled requires representation.parent_child.include_parent_records=true"
            )
        for key in ("parent_top_k", "child_top_k", "rrf_k"):
            positive(f"retrieval.parent_child.{key}", parent_retrieval.get(key))
        for key in ("parent_weight", "child_weight"):
            if float(parent_retrieval.get(key, 0.0)) < 0:
                raise ValueError(f"retrieval.parent_child.{key} must be non-negative")

    temporal = cfg.get("temporal", {})
    if temporal.get("missing_date_policy", "error") not in {"error", "exclude", "include"}:
        raise ValueError("temporal.missing_date_policy must be error, exclude, or include")

    if fusion == "weighted":
        normalization = cfg.get("fusion", {}).get("score_normalization", "minmax")
        if normalization != "minmax":
            raise ValueError("Only fusion.score_normalization=minmax is currently supported")
    fusion_cfg = cfg.get("fusion", {})
    weights = [float(fusion_cfg.get("sparse_weight", 0.0)), float(fusion_cfg.get("dense_weight", 0.0))]
    if any(weight < 0 for weight in weights) or (method == "hybrid" and not any(weights)):
        raise ValueError("Fusion weights must be non-negative and at least one must be positive")

    if rerank.get("enabled") and rerank.get("aggregation", "weighted") not in {"weighted", "rrf"}:
        raise ValueError("reranking.aggregation must be weighted or rrf")

    reproducibility_mode = cfg.get("runtime", {}).get("reproducibility_mode", "development")
    if reproducibility_mode not in {"development", "final"}:
        raise ValueError("runtime.reproducibility_mode must be development or final")
    if reproducibility_mode == "final":
        if dense_enabled:
            _require_hf_commit("indexing.dense.revision", dense_cfg.get("revision"))
        if rerank.get("enabled"):
            for i, spec in enumerate(rerank.get("models", [])):
                _require_hf_commit(f"reranking.models[{i}].revision", spec.get("revision"))

### 1.6 Corpus → retrieval records — `corpus.py`

In [9]:
def load_enriched_units(cfg: dict[str, Any]) -> list[dict[str, Any]]:
    corpus_cfg = cfg["corpus"]
    pattern_path = resolve_project_path(corpus_cfg["input_glob"])
    assert pattern_path is not None
    files = sorted(glob.glob(str(pattern_path)))
    if not files:
        raise FileNotFoundError(f"No enriched JSON files match: {pattern_path}")

    required = corpus_cfg.get("required_fields", [])
    include_types = set(corpus_cfg.get("include_unit_types", []))
    duplicate_policy = corpus_cfg.get("duplicate_unit_id_policy", "keep")
    texts_by_id: dict[str, list[str]] = defaultdict(list)
    dropped = renamed = 0
    units: list[dict[str, Any]] = []
    for file in files:
        data = json.loads(Path(file).read_text(encoding="utf-8"))
        if not isinstance(data, list):
            raise ValueError(f"Expected JSON list in {file}")
        for row in data:
            missing = [field for field in required if field not in row]
            if missing:
                raise ValueError(f"{file}: record missing fields {missing}")
            if include_types and row.get("unit_type") not in include_types:
                continue
            if duplicate_policy != "keep":
                seen_texts = texts_by_id[row["unit_id"]]
                if row["text"] in seen_texts:
                    dropped += 1
                    continue
                if seen_texts:
                    if duplicate_policy == "error":
                        raise ValueError(f"{file}: duplicate unit_id {row['unit_id']!r} with different text")
                    # Distinct record, same legal unit for evaluation (source_unit_ids).
                    row = {**row, "_record_id": f"{row['unit_id']}#dup{len(seen_texts) + 1}"}
                    renamed += 1
                seen_texts.append(row["text"])
            units.append(row)
    if dropped or renamed:
        logger.info("Duplicate unit_id: dropped %d identical rows, kept %d differing rows as #dupN records",
                    dropped, renamed)
    return units


def build_index_text(unit: dict[str, Any], rep_cfg: dict[str, Any]) -> str:
    return _build_text_from_fields(unit, rep_cfg.get("fields", []), rep_cfg.get("separator", "\n"))


def _build_text_from_fields(unit: dict[str, Any], fields: list[dict[str, Any]], separator: str = "\n") -> str:
    parts: list[str] = []
    for spec in fields:
        if not spec.get("enabled", True):
            continue
        value = unit.get(spec["name"])
        if value is None or str(value).strip() == "":
            continue
        parts.append(f"{spec.get('prefix', '')}{normalize_whitespace(str(value))}")
    return separator.join(parts)


def _metadata_from_unit(unit: dict[str, Any]) -> dict[str, Any]:
    return {k: v for k, v in unit.items() if k not in {"text", "text_folded", "_record_id"}}


def _hierarchical_records(units: list[dict[str, Any]], cfg: dict[str, Any]) -> list[RetrievalRecord]:
    rep_cfg = cfg["representation"]
    qp = cfg.get("query_processing", {})
    records: list[RetrievalRecord] = []
    for unit in units:
        text = build_index_text(unit, rep_cfg)
        records.append(
            RetrievalRecord(
                record_id=unit.get("_record_id", unit["unit_id"]),
                text=text,
                sparse_text=sparse_normalize(
                    text,
                    lowercase=qp.get("lowercase_for_sparse", True),
                    fold=qp.get("fold_vietnamese_for_sparse", True),
                ),
                source_unit_ids=[unit["unit_id"]],
                metadata=_metadata_from_unit(unit),
            )
        )
    return records


def _windows(words: list[tuple[str, str]], size: int, overlap: int):
    if size <= 0:
        raise ValueError("chunk_size must be > 0")
    if overlap < 0 or overlap >= size:
        raise ValueError("overlap must satisfy 0 <= overlap < chunk_size")
    step = size - overlap
    for start in range(0, len(words), step):
        chunk = words[start:start + size]
        if chunk:
            yield start, chunk
        if start + size >= len(words):
            break


def _fixed_length_records(units: list[dict[str, Any]], cfg: dict[str, Any]) -> list[RetrievalRecord]:
    rep_cfg = cfg["representation"]
    fixed = rep_cfg["fixed_length"]
    if fixed.get("unit", "words") != "words":
        raise ValueError("Current fixed_length implementation supports unit=words")
    allowed_types = set(fixed.get("source_unit_types", []))
    size = int(fixed.get("chunk_size", 220))
    overlap = int(fixed.get("overlap", 40))
    qp = cfg.get("query_processing", {})

    by_doc: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for unit in units:
        if not allowed_types or unit.get("unit_type") in allowed_types:
            by_doc[unit["doc_id"]].append(unit)

    records: list[RetrievalRecord] = []
    for doc_id, doc_units in by_doc.items():
        stream: list[tuple[str, str]] = []
        for unit in doc_units:
            text = build_index_text(unit, rep_cfg)
            # Attach each word to the original legal unit for evaluation mapping.
            stream.extend((word, unit["unit_id"]) for word in text.split())
        for idx, (_, chunk) in enumerate(_windows(stream, size, overlap), start=1):
            chunk_text = " ".join(word for word, _ in chunk)
            source_ids = list(dict.fromkeys(uid for _, uid in chunk))
            representative = next((u for u in doc_units if u["unit_id"] == source_ids[0]), doc_units[0])
            metadata = _metadata_from_unit(representative)
            metadata.update({
                "doc_id": doc_id,
                "unit_type": "fixed_chunk",
                "chunk_index": idx,
                "source_unit_count": len(source_ids),
            })
            rid = f"{doc_id}__fixedchunk-{idx:05d}"
            records.append(
                RetrievalRecord(
                    record_id=rid,
                    text=chunk_text,
                    sparse_text=sparse_normalize(
                        chunk_text,
                        lowercase=qp.get("lowercase_for_sparse", True),
                        fold=qp.get("fold_vietnamese_for_sparse", True),
                    ),
                    source_unit_ids=source_ids,
                    metadata=metadata,
                )
            )
    return records


def _hierarchy_key(unit: dict[str, Any]) -> tuple[str, str, str, str]:
    '''Identify an article version without parsing the display-oriented unit_id.'''
    return (
        str(unit.get("doc_id", "")),
        str(unit.get("article", "")),
        str(unit.get("effective_from") or ""),
        str(unit.get("effective_to") or ""),
    )


def _short_parent_heading(parent: dict[str, Any], max_words: int) -> str:
    # Workstream A article text starts with the article number and heading. Some
    # rows also contain a short lead sentence, so cap it rather than copying a
    # whole parent body into every child.
    raw = normalize_whitespace(str(parent.get("text") or ""))
    if not raw:
        raw = normalize_whitespace(str(parent.get("title_chain") or "").split(">")[-1])
    words = raw.split()
    return " ".join(words[:max_words])


def _parent_child_records(units: list[dict[str, Any]], cfg: dict[str, Any]) -> list[RetrievalRecord]:
    rep_cfg = cfg["representation"]
    pcfg = rep_cfg["parent_child"]
    qp = cfg.get("query_processing", {})
    separator = rep_cfg.get("separator", "\n")
    parent_type = pcfg.get("parent_unit_type", "article")
    child_types = set(pcfg.get("child_unit_types", ["clause", "point"]))
    child_fields = pcfg.get("child_fields", [{"name": "text", "prefix": "[TEXT] "}])
    parent_fields = pcfg.get("parent_fields", rep_cfg.get("fields", []))
    context_cfg = pcfg.get("context", {})
    include_heading = bool(context_cfg.get("include_parent_heading", False))
    max_heading_words = int(context_cfg.get("parent_heading_max_words", 40))
    orphan_policy = pcfg.get("orphan_policy", "error")

    parents = {_hierarchy_key(unit): unit for unit in units if unit.get("unit_type") == parent_type}
    records: list[RetrievalRecord] = []

    if pcfg.get("include_parent_records", False):
        for parent in parents.values():
            text = _build_text_from_fields(parent, parent_fields, separator)
            metadata = _metadata_from_unit(parent)
            metadata.update({
                "retrieval_role": "parent",
                "parent_id": None,
                "article_id": parent["unit_id"],
            })
            records.append(RetrievalRecord(
                record_id=parent["unit_id"],
                text=text,
                sparse_text=sparse_normalize(
                    text,
                    lowercase=qp.get("lowercase_for_sparse", True),
                    fold=qp.get("fold_vietnamese_for_sparse", True),
                ),
                source_unit_ids=[parent["unit_id"]],
                metadata=metadata,
            ))

    for child in units:
        if child.get("unit_type") not in child_types:
            continue
        parent = parents.get(_hierarchy_key(child))
        if parent is None:
            if orphan_policy == "skip":
                continue
            raise ValueError(
                "Cannot resolve article parent for child "
                f"{child.get('unit_id')!r} using doc/article/effective interval"
            )

        child_text = _build_text_from_fields(child, child_fields, separator)
        heading = _short_parent_heading(parent, max_heading_words)
        text = separator.join(
            part for part in (
                f"{context_cfg.get('parent_prefix', '[PARENT] ')}{heading}" if include_heading and heading else "",
                child_text,
            ) if part
        )
        metadata = _metadata_from_unit(child)
        metadata.update({
            "retrieval_role": "child",
            "parent_id": parent["unit_id"],
            "article_id": parent["unit_id"],
            "parent_heading": heading,
        })
        records.append(RetrievalRecord(
            record_id=child.get("_record_id", child["unit_id"]),
            text=text,
            sparse_text=sparse_normalize(
                text,
                lowercase=qp.get("lowercase_for_sparse", True),
                fold=qp.get("fold_vietnamese_for_sparse", True),
            ),
            source_unit_ids=[child["unit_id"]],
            metadata=metadata,
        ))
    return records


def build_records(cfg: dict[str, Any]) -> list[RetrievalRecord]:
    units = load_enriched_units(cfg)
    mode = cfg["representation"]["mode"]
    if mode == "hierarchical_units":
        return _hierarchical_records(units, cfg)
    if mode == "fixed_length":
        return _fixed_length_records(units, cfg)
    if mode == "parent_child":
        return _parent_child_records(units, cfg)
    raise ValueError(mode)

### 1.7 Query processing — `query.py`

In [10]:
@dataclass
class ProcessedQuery:
    raw: str
    dense: str
    sparse: str
    filters: dict[str, Any]
    expansions: list[str]
    temporal: TemporalConstraint | None = None


class QueryProcessor:
    def __init__(self, cfg: dict[str, Any]):
        self.cfg = cfg.get("query_processing", {})
        self.temporal_cfg = cfg.get("temporal", {})

    def process(self, query: str, filters: dict[str, Any] | None = None) -> ProcessedQuery:
        normalized = normalize_whitespace(query) if self.cfg.get("normalize_whitespace", True) else query
        temporal_cfg = self.temporal_cfg
        temporal = parse_temporal_query(normalized, temporal_cfg) if temporal_cfg.get("enabled") else None
        semantic_query = temporal.semantic_query if temporal else normalized
        expansion_cfg = self.cfg.get("query_expansion", {})
        expansions: list[str] = []
        if expansion_cfg.get("enabled") and expansion_cfg.get("method") == "static_map":
            lower = semantic_query.lower()
            for trigger, terms in expansion_cfg.get("static_map", {}).items():
                if trigger.lower() in lower:
                    expansions.extend(str(x) for x in terms)
        # Stable deduplication.
        expansions = list(dict.fromkeys(expansions))
        expanded = semantic_query if not expansions else semantic_query + " " + " ".join(expansions)
        sparse = sparse_normalize(
            expanded,
            lowercase=self.cfg.get("lowercase_for_sparse", True),
            fold=self.cfg.get("fold_vietnamese_for_sparse", True),
        )
        filter_cfg = self.cfg.get("metadata_filter", {})
        applied_filters = (filters or {}) if filter_cfg.get("enabled", True) else {}
        return ProcessedQuery(
            raw=query,
            dense=expanded,
            sparse=sparse,
            filters=applied_filters,
            expansions=expansions,
            temporal=temporal,
        )

### 1.8 Dense encoder — `embedding.py`

In [11]:
def resolve_device(device: str) -> str:
    if device != "auto":
        return device
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        return "cpu"


_ST_MODEL_CACHE: dict[tuple, Any] = {}


class SentenceTransformerEncoder:
    def __init__(self, cfg: dict[str, Any]):
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError as exc:
            raise ImportError(
                "Dense retrieval/reranking dependencies are not installed. "
                "Run `pip install sentence-transformers faiss-cpu`."
            ) from exc
        self.cfg = cfg
        self.device = resolve_device(cfg.get("device", "auto"))
        key = (cfg["model_name"], cfg.get("revision"), self.device, cfg.get("trust_remote_code", False))
        if key not in _ST_MODEL_CACHE:
            _ST_MODEL_CACHE[key] = SentenceTransformer(
                cfg["model_name"],
                device=self.device,
                trust_remote_code=cfg.get("trust_remote_code", False),
                revision=cfg.get("revision"),
            )
        self.model = _ST_MODEL_CACHE[key]

    def encode_documents(self, texts: list[str]) -> np.ndarray:
        prefix = self.cfg.get("document_prefix", "")
        payload = [prefix + t for t in texts]
        return np.asarray(self.model.encode(
            payload,
            batch_size=int(self.cfg.get("batch_size", 8)),
            normalize_embeddings=bool(self.cfg.get("normalize_embeddings", True)),
            show_progress_bar=True,
            convert_to_numpy=True,
        ), dtype="float32")

    def encode_query(self, text: str) -> np.ndarray:
        prefix = self.cfg.get("query_prefix", "")
        return np.asarray(self.model.encode(
            [prefix + text],
            batch_size=1,
            normalize_embeddings=bool(self.cfg.get("normalize_embeddings", True)),
            show_progress_bar=False,
            convert_to_numpy=True,
        )[0], dtype="float32")

### 1.9 Sparse index (BM25) — `indexing/sparse.py` 

In [12]:
class BM25Index:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = float(k1)
        self.b = float(b)
        self.doc_ids: list[str] = []
        self.doc_lengths: list[int] = []
        self.avgdl = 0.0
        self.term_freqs: list[Counter[str]] = []
        self.idf: dict[str, float] = {}

    def build(self, records: list[RetrievalRecord]) -> None:
        self.doc_ids = [r.record_id for r in records]
        tokenized = [tokenize(r.sparse_text) for r in records]
        self.doc_lengths = [len(tokens) for tokens in tokenized]
        self.avgdl = sum(self.doc_lengths) / max(len(self.doc_lengths), 1)
        self.term_freqs = [Counter(tokens) for tokens in tokenized]
        df: Counter[str] = Counter()
        for tf in self.term_freqs:
            df.update(tf.keys())
        n = len(records)
        self.idf = {
            term: math.log(1.0 + (n - freq + 0.5) / (freq + 0.5))
            for term, freq in df.items()
        }

    def search(self, query: str, top_k: int, allowed_record_ids: set[str] | None = None) -> list[SearchHit]:
        q_tokens = tokenize(query)
        scores: list[tuple[float, str]] = []
        for i, tf in enumerate(self.term_freqs):
            if allowed_record_ids is not None and self.doc_ids[i] not in allowed_record_ids:
                continue
            dl = self.doc_lengths[i]
            score = 0.0
            for term in q_tokens:
                freq = tf.get(term, 0)
                if not freq:
                    continue
                denom = freq + self.k1 * (1 - self.b + self.b * dl / max(self.avgdl, 1e-12))
                score += self.idf.get(term, 0.0) * (freq * (self.k1 + 1)) / denom
            if score > 0:
                scores.append((score, self.doc_ids[i]))
        scores.sort(key=lambda x: x[0], reverse=True)
        return [SearchHit(record_id=doc_id, score=score, rank=i + 1, source="sparse")
                for i, (score, doc_id) in enumerate(scores[:top_k])]

### 1.10 Dense index backends — `indexing/dense.py` (FAISS + NumPy flat)

In [13]:
class BaseDenseIndex(ABC):
    @abstractmethod
    def build(self, embeddings: np.ndarray, record_ids: list[str]) -> None: ...

    @abstractmethod
    def search(
        self,
        query_embedding: np.ndarray,
        top_k: int,
        allowed_record_ids: set[str] | None = None,
    ) -> list[SearchHit]: ...


_DENSE_BACKENDS: dict[str, type[BaseDenseIndex]] = {}


def register_dense_backend(name: str, cls: type[BaseDenseIndex]) -> None:
    _DENSE_BACKENDS[name] = cls


def create_dense_backend(name: str, cfg: dict) -> BaseDenseIndex:
    if name not in _DENSE_BACKENDS:
        raise ValueError(f"Unknown dense backend {name!r}. Registered: {sorted(_DENSE_BACKENDS)}")
    return _DENSE_BACKENDS[name](cfg)


class NumpyFlatIndex(BaseDenseIndex):
    def __init__(self, cfg: dict | None = None):
        self.cfg = cfg or {}
        self.embeddings: np.ndarray | None = None
        self.record_ids: list[str] = []

    def build(self, embeddings: np.ndarray, record_ids: list[str]) -> None:
        self.embeddings = np.asarray(embeddings, dtype="float32")
        self.record_ids = list(record_ids)

    def search(
        self,
        query_embedding: np.ndarray,
        top_k: int,
        allowed_record_ids: set[str] | None = None,
    ) -> list[SearchHit]:
        if self.embeddings is None:
            raise RuntimeError("Index not built")
        q = np.asarray(query_embedding, dtype="float32").reshape(-1)
        scores = self.embeddings @ q
        eligible = np.arange(len(scores))
        if allowed_record_ids is not None:
            eligible = np.asarray(
                [i for i, record_id in enumerate(self.record_ids) if record_id in allowed_record_ids],
                dtype=int,
            )
        if len(eligible) == 0:
            return []
        k = min(top_k, len(eligible))
        eligible_scores = scores[eligible]
        local_idx = np.argpartition(-eligible_scores, k - 1)[:k]
        idx = eligible[local_idx]
        idx = idx[np.argsort(-scores[idx])]
        return [SearchHit(record_id=self.record_ids[i], score=float(scores[i]), rank=r + 1, source="dense")
                for r, i in enumerate(idx)]


class FaissDenseIndex(BaseDenseIndex):
    def __init__(self, cfg: dict | None = None):
        self.cfg = cfg or {}
        self.index = None
        self.record_ids: list[str] = []

    @staticmethod
    def _faiss():
        try:
            import faiss
            return faiss
        except ImportError as exc:
            raise ImportError(
                "FAISS backend requested but faiss-cpu is not installed. "
                "Run `pip install faiss-cpu` or set indexing.dense.backend=numpy_flat."
            ) from exc

    def build(self, embeddings: np.ndarray, record_ids: list[str]) -> None:
        faiss = self._faiss()
        x = np.ascontiguousarray(embeddings, dtype="float32")
        dim = x.shape[1]
        index_type = self.cfg.get("index_type", "FlatIP")
        if index_type == "FlatIP":
            self.index = faiss.IndexFlatIP(dim)
        elif index_type == "FlatL2":
            self.index = faiss.IndexFlatL2(dim)
        else:
            # Allows future FAISS factory strings such as HNSW32 or IVF256,Flat.
            self.index = faiss.index_factory(dim, index_type, faiss.METRIC_INNER_PRODUCT)
            if not self.index.is_trained:
                self.index.train(x)
        self.index.add(x)
        self.record_ids = list(record_ids)

    def search(
        self,
        query_embedding: np.ndarray,
        top_k: int,
        allowed_record_ids: set[str] | None = None,
    ) -> list[SearchHit]:
        if self.index is None:
            raise RuntimeError("Index not built")
        q = np.ascontiguousarray(query_embedding, dtype="float32").reshape(1, -1)
        # Flat FAISS indexes do not expose portable metadata filtering. Search
        # all records when an allow-list is present, then filter exactly.
        k = len(self.record_ids) if allowed_record_ids is not None else min(top_k, len(self.record_ids))
        if k == 0:
            return []
        scores, indices = self.index.search(q, k)
        hits = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            if allowed_record_ids is not None and self.record_ids[idx] not in allowed_record_ids:
                continue
            # SearchHit scores consistently mean "higher is better". FAISS L2
            # returns distances (lower is better), so expose their negative.
            value = -float(score) if self.cfg.get("index_type", "FlatIP") == "FlatL2" else float(score)
            hits.append(SearchHit(record_id=self.record_ids[idx], score=value, rank=len(hits) + 1, source="dense"))
            if len(hits) >= top_k:
                break
        return hits


register_dense_backend("numpy_flat", NumpyFlatIndex)
register_dense_backend("faiss", FaissDenseIndex)

### 1.11 Index manager (in-memory) — `indexing/manager.py`

In [14]:
_INDEX_CACHE: dict[str, dict[str, Any]] = {}


def clear_index_cache() -> None:
    '''Free memory (indexes + loaded embedding models).'''
    _INDEX_CACHE.clear()
    _ST_MODEL_CACHE.clear()


class IndexManager:
    def __init__(self, cfg: dict[str, Any]):
        self.cfg = cfg
        self.index_signature = self.signature()
        self.last_operation = "not_run"
        self.last_operation_seconds = 0.0
        self._artifact: dict[str, Any] | None = None

    def _signature_payload(self) -> dict[str, Any]:
        return {
            "corpus": self.cfg.get("corpus"),
            "representation": self.cfg.get("representation"),
            "query_processing": {
                k: v for k, v in self.cfg.get("query_processing", {}).items()
                if k in {"lowercase_for_sparse", "fold_vietnamese_for_sparse"}
            },
            "sparse": self.cfg.get("indexing", {}).get("sparse"),
            "dense": self.cfg.get("indexing", {}).get("dense"),
        }

    def _corpus_digest(self) -> str:
        pattern = resolve_project_path(self.cfg["corpus"]["input_glob"])
        assert pattern is not None
        h = hashlib.sha256()
        for filename in sorted(glob.glob(str(pattern))):
            path = Path(filename)
            h.update(path.name.encode("utf-8"))
            with path.open("rb") as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b""):
                    h.update(chunk)
        return h.hexdigest()

    def signature(self) -> str:
        payload = self._signature_payload()
        payload["corpus_digest"] = self._corpus_digest()
        raw = json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
        return hashlib.sha256(raw).hexdigest()

    def needs_build(self) -> bool:
        return self.index_signature not in _INDEX_CACHE

    def ensure(self) -> None:
        started = time.perf_counter()
        if self.needs_build():
            if not self.cfg.get("runtime", {}).get("auto_build_index", True) and not self.cfg.get("runtime", {}).get("force_rebuild", False):
                raise FileNotFoundError(f"Index is missing: {self.index_signature}")
            self.build()
            return
        self._artifact = _INDEX_CACHE[self.index_signature]
        self.last_operation = "reused"
        self.last_operation_seconds = time.perf_counter() - started

    def build(self) -> None:
        started = time.perf_counter()
        if not self.needs_build():
            logger.info("Index already in memory; reusing %s", self.index_signature[:12])
            self._artifact = _INDEX_CACHE[self.index_signature]
            self.last_operation = "reused"
            self.last_operation_seconds = time.perf_counter() - started
            return

        logger.info("Building retrieval records...")
        records = build_records(self.cfg)
        logger.info("Built %d retrieval records", len(records))
        artifact: dict[str, Any] = {"records": records, "sparse": None, "dense": None}

        sparse_cfg = self.cfg["indexing"]["sparse"]
        if sparse_cfg.get("enabled"):
            logger.info("Building BM25 index...")
            index = BM25Index(k1=sparse_cfg.get("k1", 1.5), b=sparse_cfg.get("b", 0.75))
            index.build(records)
            artifact["sparse"] = index

        dense_cfg = self.cfg["indexing"]["dense"]
        if dense_cfg.get("enabled"):
            logger.info("Encoding %d records with %s...", len(records), dense_cfg["model_name"])
            encoder = SentenceTransformerEncoder(dense_cfg)
            embeddings = encoder.encode_documents([r.text for r in records])
            backend = create_dense_backend(dense_cfg.get("backend", "faiss"), dense_cfg)
            backend.build(embeddings, [r.record_id for r in records])
            artifact["dense"] = backend

        artifact["manifest"] = {
            "signature": self.index_signature,
            "record_count": len(records),
            "representation_mode": self.cfg["representation"]["mode"],
            "sparse_enabled": bool(sparse_cfg.get("enabled")),
            "dense_enabled": bool(dense_cfg.get("enabled")),
            "dense_backend": dense_cfg.get("backend") if dense_cfg.get("enabled") else None,
            "dense_model": dense_cfg.get("model_name") if dense_cfg.get("enabled") else None,
        }
        _INDEX_CACHE[self.index_signature] = artifact
        self._artifact = artifact
        self.last_operation = "built"
        self.last_operation_seconds = time.perf_counter() - started
        logger.info("Index ready in memory (%s)", self.index_signature[:12])

    @property
    def record_count(self) -> int:
        return self._artifact["manifest"]["record_count"] if self._artifact else 0

    def load_records(self) -> list[RetrievalRecord]:
        return list(self._artifact["records"])

    def load_sparse(self) -> BM25Index:
        return self._artifact["sparse"]

    def load_dense(self) -> BaseDenseIndex:
        return self._artifact["dense"]

### 1.12 Fusion — `fusion.py`

In [15]:
def _minmax_hits(hits: list[SearchHit]) -> dict[str, float]:
    if not hits:
        return {}
    values = [h.score for h in hits]
    lo, hi = min(values), max(values)
    if hi == lo:
        return {h.record_id: 1.0 for h in hits}
    return {h.record_id: (h.score - lo) / (hi - lo) for h in hits}


def rrf_fusion(lists: dict[str, list[SearchHit]], k: int = 60, weights: dict[str, float] | None = None) -> list[SearchHit]:
    scores = defaultdict(float)
    components: dict[str, dict[str, float]] = defaultdict(dict)
    weights = weights or {}
    for name, hits in lists.items():
        weight = float(weights.get(name, 1.0))
        for rank, hit in enumerate(hits, start=1):
            contribution = weight / (k + rank)
            scores[hit.record_id] += contribution
            components[hit.record_id][name] = hit.score
    ordered = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [SearchHit(record_id=rid, score=score, rank=i + 1, source="fusion", component_scores=components[rid])
            for i, (rid, score) in enumerate(ordered)]


def weighted_fusion(lists: dict[str, list[SearchHit]], weights: dict[str, float]) -> list[SearchHit]:
    scores = defaultdict(float)
    components: dict[str, dict[str, float]] = defaultdict(dict)
    for name, hits in lists.items():
        norm = _minmax_hits(hits)
        weight = float(weights.get(name, 1.0))
        for hit in hits:
            scores[hit.record_id] += weight * norm[hit.record_id]
            components[hit.record_id][name] = hit.score
    ordered = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [SearchHit(record_id=rid, score=score, rank=i + 1, source="fusion", component_scores=components[rid])
            for i, (rid, score) in enumerate(ordered)]

### 1.13 Cross-encoder reranking — `rerank.py`

In [16]:
def _minmax_values(values: list[float]) -> list[float]:
    if not values:
        return []
    lo, hi = min(values), max(values)
    if hi == lo:
        return [1.0] * len(values)
    return [(x - lo) / (hi - lo) for x in values]


class MultiCrossEncoderReranker:
    def __init__(self, cfg: dict[str, Any]):
        self.cfg = cfg
        self._models = []

    def _load(self):
        if self._models:
            return
        try:
            from sentence_transformers import CrossEncoder
        except ImportError as exc:
            raise ImportError("Reranking requires sentence-transformers.") from exc
        for spec in self.cfg.get("models", []):
            model = CrossEncoder(
                spec["model_name"],
                device=resolve_device(spec.get("device", "auto")),
                trust_remote_code=spec.get("trust_remote_code", False),
                revision=spec.get("revision"),
            )
            self._models.append((spec, model))

    def rerank(self, query: str, hits: list[SearchHit], records: dict[str, RetrievalRecord]) -> list[SearchHit]:
        self._load()
        top_n = min(int(self.cfg.get("top_n", 30)), len(hits))
        head, tail = hits[:top_n], hits[top_n:]
        if not head:
            return hits
        pairs = [(query, records[h.record_id].text) for h in head]
        aggregation = self.cfg.get("aggregation", "weighted")

        per_model_hits: dict[str, list[SearchHit]] = {}
        weighted_scores = [0.0] * len(head)
        for model_index, (spec, model) in enumerate(self._models):
            raw = model.predict(pairs, batch_size=int(self.cfg.get("batch_size", 8)), show_progress_bar=False)
            scores = [float(x) for x in raw]
            name = spec.get("name") or spec["model_name"]
            if aggregation == "rrf":
                order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
                per_model_hits[name] = [SearchHit(record_id=head[i].record_id, score=scores[i], rank=r + 1, source=name)
                                        for r, i in enumerate(order)]
            else:
                norm = _minmax_values(scores)
                weight = float(spec.get("weight", 1.0))
                for i, value in enumerate(norm):
                    weighted_scores[i] += weight * value
                    head[i].component_scores[f"reranker:{name}"] = scores[i]

        if aggregation == "rrf":
            fused = rrf_fusion(per_model_hits, k=int(self.cfg.get("rrf_k", 60)))
            id_to_original = {h.record_id: h for h in head}
            for h in fused:
                h.record = id_to_original[h.record_id].record
                h.component_scores.update(id_to_original[h.record_id].component_scores)
                h.source = "reranker_rrf"
            reranked = fused
        else:
            for h, score in zip(head, weighted_scores):
                h.score = score
                h.source = "reranker"
            reranked = sorted(head, key=lambda h: h.score, reverse=True)

        output = reranked + tail
        for rank, hit in enumerate(output, start=1):
            hit.rank = rank
        return output

### 1.14 Retrieval pipeline — `pipeline.py`

In [17]:
class RetrievalPipeline:
    def __init__(self, cfg: dict[str, Any]):
        self.cfg = cfg
        self.manager = IndexManager(cfg)
        self.manager.ensure()
        records = self.manager.load_records()
        self.records: dict[str, RetrievalRecord] = {r.record_id: r for r in records}
        self.query_processor = QueryProcessor(cfg)
        self.sparse = None
        self.dense = None
        self.encoder = None
        method = cfg["retrieval"]["method"]
        if method in {"sparse", "hybrid"}:
            self.sparse = self.manager.load_sparse()
        if method in {"dense", "hybrid"}:
            self.dense = self.manager.load_dense()
            self.encoder = SentenceTransformerEncoder(cfg["indexing"]["dense"])
        self.reranker = MultiCrossEncoderReranker(cfg["reranking"]) if cfg.get("reranking", {}).get("enabled") else None
        self.last_query_context: dict[str, Any] = {}

    @staticmethod
    def _matches_filters(record: RetrievalRecord, filters: dict[str, Any]) -> bool:
        for key, expected in filters.items():
            actual = record.metadata.get(key)
            if isinstance(expected, list):
                if actual not in expected:
                    return False
            elif actual != expected:
                return False
        return True

    def _filter_hits(self, hits: list[SearchHit], filters: dict[str, Any], limit: int) -> list[SearchHit]:
        if not filters:
            return hits[:limit]
        out = [h for h in hits if self._matches_filters(self.records[h.record_id], filters)]
        return out[:limit]

    def _attach_records(self, hits: list[SearchHit]) -> list[SearchHit]:
        for hit in hits:
            hit.record = self.records[hit.record_id]
        return hits

    def _eligible_record_ids(self, processed) -> set[str]:
        temporal_cfg = self.cfg.get("temporal", {})
        missing_policy = temporal_cfg.get("missing_date_policy", "exclude")
        eligible: set[str] = set()
        for record_id, record in self.records.items():
            if not self._matches_filters(record, processed.filters):
                continue
            if processed.temporal and not record_matches_temporal(record.metadata, processed.temporal, missing_policy):
                continue
            eligible.add(record_id)
        return eligible

    def _candidate_hits(
        self,
        processed,
        allowed_record_ids: set[str],
        sparse_k: int,
        dense_k: int,
        candidate_k: int,
        query_vector=None,
    ) -> list[SearchHit]:
        method = self.cfg["retrieval"]["method"]
        lists: dict[str, list[SearchHit]] = {}
        if method in {"sparse", "hybrid"}:
            lists["sparse"] = self.sparse.search(processed.sparse, sparse_k, allowed_record_ids)
        if method in {"dense", "hybrid"}:
            qvec = query_vector if query_vector is not None else self.encoder.encode_query(processed.dense)
            lists["dense"] = self.dense.search(qvec, dense_k, allowed_record_ids)

        if method == "sparse":
            return lists["sparse"][:candidate_k]
        if method == "dense":
            return lists["dense"][:candidate_k]

        fcfg = self.cfg["fusion"]
        if fcfg["method"] == "rrf":
            hits = rrf_fusion(
                lists,
                k=int(fcfg.get("rrf_k", 60)),
                weights={"sparse": fcfg.get("sparse_weight", 1.0), "dense": fcfg.get("dense_weight", 1.0)},
            )
        elif fcfg["method"] == "weighted":
            hits = weighted_fusion(
                lists,
                weights={"sparse": fcfg.get("sparse_weight", 0.5), "dense": fcfg.get("dense_weight", 0.5)},
            )
        else:
            raise ValueError(fcfg["method"])
        return hits[:candidate_k]

    def _parent_to_child_hits(self, processed, eligible: set[str], query_vector=None) -> list[SearchHit]:
        pcfg = self.cfg["retrieval"]["parent_child"]
        parent_ids = {
            record_id for record_id in eligible
            if self.records[record_id].metadata.get("retrieval_role") == "parent"
        }
        parent_top_k = int(pcfg.get("parent_top_k", 10))
        parents = self._candidate_hits(
            processed,
            parent_ids,
            parent_top_k,
            parent_top_k,
            parent_top_k,
            query_vector,
        )
        parent_ranks = {hit.record_id: rank for rank, hit in enumerate(parents, start=1)}
        parent_scores = {hit.record_id: hit.score for hit in parents}

        child_ids = {
            record_id for record_id in eligible
            if self.records[record_id].metadata.get("retrieval_role") == "child"
            and self.records[record_id].metadata.get("parent_id") in parent_ranks
        }
        child_top_k = int(pcfg.get("child_top_k", self.cfg["retrieval"].get("candidate_top_k", 50)))
        children = self._candidate_hits(
            processed,
            child_ids,
            child_top_k,
            child_top_k,
            child_top_k,
            query_vector,
        )

        rrf_k = int(pcfg.get("rrf_k", 60))
        parent_weight = float(pcfg.get("parent_weight", 0.7))
        child_weight = float(pcfg.get("child_weight", 1.0))
        for child_rank, hit in enumerate(children, start=1):
            parent_id = self.records[hit.record_id].metadata["parent_id"]
            parent_rank = parent_ranks[parent_id]
            child_rrf = child_weight / (rrf_k + child_rank)
            parent_rrf = parent_weight / (rrf_k + parent_rank)
            hit.component_scores.update({
                "child_raw": hit.score,
                "child_rrf": child_rrf,
                "parent_raw": parent_scores[parent_id],
                "parent_rrf": parent_rrf,
            })
            hit.score = child_rrf + parent_rrf
            hit.source = "parent_to_child"
        children.sort(key=lambda hit: hit.score, reverse=True)
        return children

    def _deduplicate(self, hits: list[SearchHit]) -> list[SearchHit]:
        mode = self.cfg.get("ranking", {}).get("deduplicate_by", "source_unit")
        if mode == "none":
            return hits
        seen = set()
        out = []
        for hit in hits:
            if mode == "record_id":
                key = hit.record_id
            elif mode == "source_unit":
                # For fixed chunks, use the first source legal unit as the primary dedupe key.
                key = hit.record.source_unit_ids[0] if hit.record and hit.record.source_unit_ids else hit.record_id
            else:
                raise ValueError(f"Unknown ranking.deduplicate_by={mode}")
            if key in seen:
                continue
            seen.add(key)
            out.append(hit)
        return out

    def retrieve(self, query: str, top_k: int | None = None, filters: dict[str, Any] | None = None) -> list[SearchHit]:
        processed = self.query_processor.process(query, filters)
        rcfg = self.cfg["retrieval"]
        method = rcfg["method"]
        eligible = self._eligible_record_ids(processed)
        self.last_query_context = {
            "temporal": processed.temporal.to_dict() if processed.temporal else None,
            "eligible_record_count": len(eligible),
            "filters": processed.filters,
        }
        query_vector = self.encoder.encode_query(processed.dense) if method in {"dense", "hybrid"} else None
        candidate_k = int(rcfg.get("candidate_top_k", 50))
        pcfg = rcfg.get("parent_child", {})
        if pcfg.get("enabled"):
            hits = self._parent_to_child_hits(processed, eligible, query_vector)
        else:
            hits = self._candidate_hits(
                processed,
                eligible,
                int(rcfg.get("sparse_top_k", 50)),
                int(rcfg.get("dense_top_k", 50)),
                candidate_k,
                query_vector,
            )
        hits = self._attach_records(hits)

        if self.reranker:
            hits = self.reranker.rerank(processed.dense, hits, self.records)

        hits = self._deduplicate(hits)
        final_k = int(self.cfg.get("ranking", {}).get("final_top_k", 10) if top_k is None else top_k)
        if final_k <= 0:
            raise ValueError("top_k must be > 0")
        hits = hits[:final_k]
        for rank, hit in enumerate(hits, start=1):
            hit.rank = rank
        return hits

### 1.15 Evaluation — `evaluation.py`

In [18]:
@dataclass
class BenchmarkQuery:
    query_id: str
    query: str
    relevant_ids: list[str]
    filters: dict[str, Any]


def load_benchmark(cfg: dict[str, Any]) -> list[BenchmarkQuery]:
    ecfg = cfg["evaluation"]
    raw_path = ecfg.get("benchmark_path")
    if not raw_path:
        return []
    path = resolve_project_path(raw_path)
    assert path is not None
    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == ".jsonl":
        rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    elif path.suffix.lower() == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        rows = payload if isinstance(payload, list) else payload.get("queries", [])
    elif path.suffix.lower() == ".csv":
        with path.open("r", encoding="utf-8-sig", newline="") as f:
            rows = list(csv.DictReader(f))
    else:
        raise ValueError("benchmark_path must be .jsonl, .json, or .csv")

    qf = ecfg.get("query_field", "query")
    qid = ecfg.get("query_id_field", "query_id")
    rf = ecfg.get("relevant_ids_field", "relevant_unit_ids")
    ff = ecfg.get("filters_field", "filters")
    out = []
    for i, row in enumerate(rows):
        relevant = row.get(rf, []) or []
        filters = row.get(ff, {}) or {}
        if isinstance(relevant, str):
            try:
                parsed = json.loads(relevant)
                relevant = parsed if isinstance(parsed, list) else [relevant]
            except json.JSONDecodeError:
                relevant = [x.strip() for x in relevant.split("|") if x.strip()]
        if isinstance(filters, str):
            filters = json.loads(filters) if filters.strip() else {}
        out.append(BenchmarkQuery(
            query_id=str(row.get(qid, i)),
            query=str(row[qf]),
            relevant_ids=[str(x) for x in relevant],
            filters=filters,
        ))
    return out


def build_unit_to_article(cfg: dict[str, Any]) -> dict[str, str]:
    '''Map every legal unit to its Article version (doc/article/effective interval).'''
    return {u["unit_id"]: "|".join(_hierarchy_key(u)) for u in load_enriched_units(cfg)}


def _hit_relevant_ids(hit: SearchHit, match_by: str, unit_to_article: dict[str, str] | None = None) -> set[str]:
    if match_by == "article":
        assert unit_to_article is not None, "article matching needs unit_to_article"
        unit_ids = hit.record.source_unit_ids if hit.record else [hit.record_id]
        return {unit_to_article[u] for u in unit_ids if u in unit_to_article}
    if match_by == "record_id":
        return {hit.record_id}
    if match_by == "source_unit":
        if hit.record:
            return set(hit.record.source_unit_ids)
        return {hit.record_id}
    if match_by == "doc_id":
        if hit.record:
            value = hit.record.metadata.get("doc_id")
            return {str(value)} if value is not None else set()
        return set()
    raise ValueError(f"Unknown evaluation.match_by={match_by}")


def relevance_trace(
    hits: list[SearchHit], relevant: set[str], match_by: str, unit_to_article: dict[str, str] | None = None
) -> tuple[list[int], list[int]]:
    # Overlapping chunks can point to the same legal unit. Count a relevant
    # entity only on its first occurrence so metrics cannot be inflated by
    # returning several chunks that cover the same ground truth.
    seen: set[str] = set()
    binary: list[int] = []
    newly_covered: list[int] = []
    for hit in hits:
        matched = (_hit_relevant_ids(hit, match_by, unit_to_article) & relevant) - seen
        binary.append(1 if matched else 0)
        newly_covered.append(len(matched))
        seen.update(matched)
    return binary, newly_covered


def binary_relevance(hits: list[SearchHit], relevant: set[str], match_by: str) -> list[int]:
    return relevance_trace(hits, relevant, match_by)[0]


def metrics_at_k(
    binary: list[int], relevant_count: int, k: int, newly_covered: list[int] | None = None
) -> dict[str, float]:
    rel = binary[:k]
    retrieved_relevant = sum(rel)
    precision = retrieved_relevant / k if k else 0.0
    covered = sum((newly_covered or binary)[:k])
    recall = min(covered / relevant_count, 1.0) if relevant_count else 0.0
    hit_rate = 1.0 if retrieved_relevant else 0.0
    rr = next((1.0 / (i + 1) for i, x in enumerate(rel) if x), 0.0)
    dcg = sum(x / math.log2(i + 2) for i, x in enumerate(rel))
    ideal_n = min(relevant_count, k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_n))
    ndcg = min(dcg / idcg, 1.0) if idcg else 0.0
    running = 0
    ap_sum = 0.0
    for i, x in enumerate(rel, start=1):
        if x:
            running += 1
            ap_sum += running / i
    ap = min(ap_sum / min(relevant_count, k), 1.0) if relevant_count else 0.0
    return {
        "hit_rate": hit_rate,
        "precision": precision,
        "recall": recall,
        "mrr": rr,
        "ndcg": ndcg,
        "map": ap,
    }


def evaluate_query(
    hits: list[SearchHit], relevant_ids: list[str], cfg: dict[str, Any], unit_to_article: dict[str, str] | None = None
) -> dict[str, float]:
    '''Primary match_by gives unprefixed keys (recall@10); also_match_by levels are
    prefixed (article.recall@10).'''
    ecfg = cfg["evaluation"]
    primary = ecfg.get("match_by", "source_unit")
    selected = set(ecfg.get("metrics", []))
    out: dict[str, float] = {}
    for level in dict.fromkeys([primary] + list(ecfg.get("also_match_by", []))):
        relevant = set(relevant_ids)
        if level == "article":
            assert unit_to_article is not None, "article matching needs unit_to_article"
            relevant = {unit_to_article[u] for u in relevant if u in unit_to_article}
        binary, newly_covered = relevance_trace(hits, relevant, level, unit_to_article)
        prefix = "" if level == primary else f"{level}."
        for k in ecfg.get("k_values", [10]):
            values = metrics_at_k(binary, len(relevant), int(k), newly_covered)
            for name, value in values.items():
                if name in selected:
                    out[f"{prefix}{name}@{k}"] = value
    return out


def aggregate_metrics(per_query: list[dict[str, float]]) -> dict[str, float]:
    if not per_query:
        return {}
    keys = sorted(set().union(*(row.keys() for row in per_query)))
    return {key: sum(row.get(key, 0.0) for row in per_query) / len(per_query) for key in keys}

### 1.16 Runtime metrics — `runtime_metrics.py`

In [19]:
def _percentile(values: list[float], percentile: float) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * percentile
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1.0 - weight) + ordered[upper] * weight


def latency_summary(samples: list[float]) -> dict[str, float | int | None]:
    if not samples:
        return {
            "count": 0,
            "total_seconds": 0.0,
            "mean_seconds": None,
            "median_seconds": None,
            "p95_seconds": None,
            "min_seconds": None,
            "max_seconds": None,
            "first_query_seconds": None,
            "warm_mean_seconds": None,
        }
    total = sum(samples)
    warm = samples[1:]
    return {
        "count": len(samples),
        "total_seconds": total,
        "mean_seconds": total / len(samples),
        "median_seconds": _percentile(samples, 0.5),
        "p95_seconds": _percentile(samples, 0.95),
        "min_seconds": min(samples),
        "max_seconds": max(samples),
        # The first query may include lazy reranker/model initialization.
        "first_query_seconds": samples[0],
        "warm_mean_seconds": sum(warm) / len(warm) if warm else None,
    }


def reset_gpu_peak_memory() -> None:
    try:
        import torch

        if not torch.cuda.is_available():
            return
        for device_index in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(device_index)
    except (ImportError, RuntimeError):
        return


def synchronize_gpu() -> None:
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.synchronize()
    except (ImportError, RuntimeError):
        return


def resource_usage() -> dict[str, Any]:
    peak_ram_bytes: int | None = None
    try:
        import resource

        peak_rss = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        # Linux reports KiB; macOS reports bytes.
        peak_ram_bytes = peak_rss if platform.system() == "Darwin" else peak_rss * 1024
    except (ImportError, OSError, ValueError):
        pass

    gpu_devices: list[dict[str, Any]] = []
    try:
        import torch

        if torch.cuda.is_available():
            synchronize_gpu()
            for device_index in range(torch.cuda.device_count()):
                gpu_devices.append({
                    "index": device_index,
                    "name": torch.cuda.get_device_name(device_index),
                    "peak_allocated_bytes": int(torch.cuda.max_memory_allocated(device_index)),
                    "peak_reserved_bytes": int(torch.cuda.max_memory_reserved(device_index)),
                })
    except (ImportError, RuntimeError):
        pass

    return {
        "peak_process_ram_bytes": peak_ram_bytes,
        "cuda_available": bool(gpu_devices),
        "gpu_devices": gpu_devices,
    }

### 1.17 Experiment runner — `experiment.py`

In [ ]:
# Metrics shown in reports and per-query rows; the full grid (all metrics x all
# k_values) is kept only in summary.json.
HEADLINE_K = 10
HEADLINE_METRICS = ("recall", "mrr", "ndcg")
SNIPPET_CHARS = 160


def _run_id() -> str:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    return f"{stamp}-{uuid.uuid4().hex[:8]}"


def _seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def _package_versions() -> dict[str, str]:
    names = ("numpy", "PyYAML", "sentence-transformers", "torch", "faiss-cpu")
    versions = {}
    for name in names:
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            continue
    return versions


def _git_revision() -> str | None:
    try:
        return subprocess.run(
            ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True, timeout=2,
            cwd=PROJECT_ROOT,
        ).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        return None


def _model_revisions(cfg: dict[str, Any]) -> dict[str, Any]:
    dense = cfg.get("indexing", {}).get("dense", {})
    rerankers = cfg.get("reranking", {}).get("models", [])
    return {
        "dense": {"model_name": dense.get("model_name"), "revision": dense.get("revision")}
        if dense.get("enabled") else None,
        "rerankers": [{"model_name": s.get("model_name"), "revision": s.get("revision")} for s in rerankers]
        if cfg.get("reranking", {}).get("enabled") else [],
    }


def _environment(cfg: dict[str, Any]) -> dict[str, Any]:
    return {
        "seed": int(cfg.get("project", {}).get("seed", 42)),
        "reproducibility_mode": cfg.get("runtime", {}).get("reproducibility_mode", "development"),
        "temporal_reference_date": cfg.get("temporal", {}).get("reference_date"),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "git_revision": _git_revision(),
        "packages": _package_versions(),
        "models": _model_revisions(cfg),
    }


def _baseline_id(cfg: dict[str, Any]) -> str | None:
    path = cfg.get("_baseline_config_path")
    if not path or not Path(path).exists():
        return None
    return (yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}).get("hypothesis", {}).get("id")


def _split_metric(key: str) -> tuple[str, str, int]:
    '''"article.recall@10" -> ("article", "recall", 10); unprefixed keys are "strict".'''
    level, _, rest = key.rpartition(".")
    name, _, k = rest.partition("@")
    return level or "strict", name, int(k)


def metrics_table(flat: dict[str, float]) -> dict[str, dict[str, dict[str, float]]]:
    '''{"recall@10": x, "article.recall@10": y} -> {"strict": {"recall": {"@10": x}}, "article": {...}}'''
    table: dict[str, dict[str, dict[str, float]]] = defaultdict(lambda: defaultdict(dict))
    ordered = sorted(flat.items(), key=lambda kv: (_split_metric(kv[0])[0] != "strict",) + _split_metric(kv[0]))
    for key, value in ordered:
        level, name, k = _split_metric(key)
        table[level][name][f"@{k}"] = round(float(value), 4)
    return {level: dict(names) for level, names in table.items()}


def headline_metrics(flat: dict[str, float]) -> dict[str, float]:
    '''Subset used in reports, e.g. recall@10 / article.recall@10.'''
    return {
        key: round(float(value), 4) for key, value in flat.items()
        if _split_metric(key)[1] in HEADLINE_METRICS and _split_metric(key)[2] == HEADLINE_K
    }


def _result_rows(
    hits: list[SearchHit], relevant_ids: list[str], unit_to_article: dict[str, str] | None
) -> list[dict[str, Any]]:
    '''One small row per hit: rank, id, score, whether it covers a gold unit / gold Article, snippet.'''
    relevant = set(relevant_ids)
    gold_articles = {unit_to_article[u] for u in relevant if u in unit_to_article} if unit_to_article else set()
    rows = []
    for hit in hits:
        units = set(hit.record.source_unit_ids) if hit.record else {hit.record_id}
        row: dict[str, Any] = {
            "rank": hit.rank,
            "record_id": hit.record_id,
            "score": round(float(hit.score), 4),
            "hit": bool(units & relevant),
        }
        if unit_to_article is not None:
            row["article_hit"] = bool({unit_to_article[u] for u in units if u in unit_to_article} & gold_articles)
        # The last field of a record is its own text ([TEXT]/[CHILD]); chunks are one line.
        text = (hit.record.text if hit.record else "").split("\n")[-1]
        row["snippet"] = text[:SNIPPET_CHARS] + ("…" if len(text) > SNIPPET_CHARS else "")
        rows.append(row)
    return rows


def _first_hit_rank(results: list[dict[str, Any]], field: str) -> int | None:
    return next((r["rank"] for r in results if r.get(field)), None)


def _timing(started: float, pipeline: RetrievalPipeline, query_latencies: list[float]) -> dict[str, Any]:
    latency = latency_summary(query_latencies)

    def ms(seconds: float | None) -> float | None:
        return round(seconds * 1000, 1) if seconds is not None else None

    return {
        "total_seconds": round(time.perf_counter() - started, 2),
        "index": pipeline.manager.last_operation,   # built | reused
        "index_seconds": round(pipeline.manager.last_operation_seconds, 2),
        "query_mean_ms": ms(latency["mean_seconds"]),
        "query_p95_ms": ms(latency["p95_seconds"]),
        # The first query may include lazy model loading.
        "first_query_ms": ms(latency["first_query_seconds"]),
    }


def _resources() -> dict[str, float]:
    usage = resource_usage()
    gpu = max((d["peak_allocated_bytes"] for d in usage["gpu_devices"]), default=0)
    return {
        "peak_ram_gb": round((usage["peak_process_ram_bytes"] or 0) / 2**30, 2),
        "peak_gpu_gb": round(gpu / 2**30, 2),
    }


def _summary(cfg: dict[str, Any], run_id: str, status: str, **fields: Any) -> dict[str, Any]:
    hypothesis = cfg.get("hypothesis", {})
    config_path = cfg.get("_config_path")
    return {
        "hypothesis": hypothesis.get("id"),
        "description": hypothesis.get("description", ""),
        "baseline": _baseline_id(cfg),
        "changed": cfg.get("_hypothesis_diff", {}).get("actual_changes", []),
        "config": str(Path(config_path).relative_to(PROJECT_ROOT))
        if config_path and Path(config_path).is_relative_to(PROJECT_ROOT) else config_path,
        "run_id": run_id,
        "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "status": status,
        **fields,
    }


def run_experiment(cfg: dict[str, Any]) -> Path:
    '''Writes artifacts/runs/<H>/<run_id>/{config.yaml, summary.json, per_query.jsonl}.'''
    started = time.perf_counter()
    _seed_everything(int(cfg.get("project", {}).get("seed", 42)))
    reset_gpu_peak_memory()
    name = str(cfg.get("hypothesis", {}).get("id") or "experiment")
    run_dir = RUNS_DIR / name / _run_id()
    run_dir.mkdir(parents=True, exist_ok=True)

    clean_cfg = {k: v for k, v in cfg.items() if not k.startswith("_")}
    (run_dir / "config.yaml").write_text(
        yaml.safe_dump(clean_cfg, allow_unicode=True, sort_keys=False), encoding="utf-8"
    )
    hypothesis_diff = cfg.get("_hypothesis_diff", {})
    logger.info("Hypothesis baseline: %s", cfg.get("_baseline_config_path"))
    logger.info("Declared changes = %s", hypothesis_diff.get("declared_changes", []))
    logger.info("Actual changes = %s", hypothesis_diff.get("actual_changes", []))

    def write_summary(summary: dict[str, Any]) -> None:
        (run_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    pipeline = RetrievalPipeline(cfg)
    benchmark = load_benchmark(cfg)
    if not benchmark:
        write_summary(_summary(
            cfg, run_dir.name, "index_ready_no_benchmark",
            records=pipeline.manager.record_count,
            timing=_timing(started, pipeline, []), resources=_resources(), environment=_environment(cfg),
        ))
        logger.info("No evaluation.benchmark_path configured; index was built/loaded successfully.")
        return run_dir

    ecfg = cfg["evaluation"]
    max_k = max([int(x) for x in ecfg.get("k_values", [10])] + [int(cfg["ranking"].get("final_top_k", 10))])
    match_levels = [ecfg.get("match_by")] + list(ecfg.get("also_match_by", []))
    unit_to_article = build_unit_to_article(cfg) if "article" in match_levels else None
    rows = []
    metric_rows = []
    query_latencies: list[float] = []
    for item in benchmark:
        synchronize_gpu()
        query_started = time.perf_counter()
        hits = pipeline.retrieve(item.query, top_k=max_k, filters=item.filters)
        synchronize_gpu()
        query_latency = time.perf_counter() - query_started
        query_latencies.append(query_latency)
        context = pipeline.last_query_context
        results = _result_rows(hits, item.relevant_ids, unit_to_article)
        row: dict[str, Any] = {
            "query_id": item.query_id,
            "query": item.query,
            "relevant_unit_ids": item.relevant_ids,
            "filters": item.filters,
            "temporal": context.get("temporal"),
            "eligible_records": context.get("eligible_record_count"),
            "latency_ms": round(query_latency * 1000, 1),
            "first_hit_rank": None,
            "metrics": None,
        }
        if item.relevant_ids:
            metrics = evaluate_query(hits, item.relevant_ids, cfg, unit_to_article)
            metric_rows.append(metrics)
            row["first_hit_rank"] = {"strict": _first_hit_rank(results, "hit")}
            if unit_to_article is not None:
                row["first_hit_rank"]["article"] = _first_hit_rank(results, "article_hit")
            row["metrics"] = headline_metrics(metrics)
        row["results"] = results
        rows.append(row)

    if ecfg.get("save_per_query", True):
        write_jsonl(run_dir / "per_query.jsonl", rows)
    write_summary(_summary(
        cfg, run_dir.name, "completed",
        queries=len(benchmark),
        labeled_queries=len(metric_rows),
        records=pipeline.manager.record_count,
        metrics=metrics_table(aggregate_metrics(metric_rows)),
        timing=_timing(started, pipeline, query_latencies),
        resources=_resources(),
        environment=_environment(cfg),
    ))
    return run_dir

### 1.18 Script helpers — `scripts/run_suite.py`, `scripts/compare_experiments.py`, `scripts/retrieve.py`

In [ ]:
def needs_hf_models(cfg: dict[str, Any]) -> bool:
    return bool(cfg["indexing"]["dense"].get("enabled")) or bool(cfg.get("reranking", {}).get("enabled"))


def load_notebook_config(path: str | Path, benchmark_path: str | None = BENCHMARK_PATH) -> dict[str, Any]:
    '''load_config() + set the shared benchmark path for every hypothesis.'''
    cfg = load_config(path)
    if benchmark_path is not None:
        cfg["evaluation"]["benchmark_path"] = benchmark_path
    return cfg


def run_suite(suite_path: str | Path, skip_hf_models: bool = not RUN_DENSE) -> dict[str, Path]:
    suite_path = _resolve_config_path(suite_path)
    suite = yaml.safe_load(suite_path.read_text(encoding="utf-8")) or {}
    configs = suite.get("configs", [])
    continue_on_error = bool(suite.get("continue_on_error", False))
    if not configs:
        raise ValueError("Suite has no configs")
    results: dict[str, Path] = {}
    for entry in configs:
        cfg_path = (suite_path.parent / entry).resolve()
        try:
            cfg = load_notebook_config(cfg_path)
            hid = cfg.get("hypothesis", {}).get("id")
            if skip_hf_models and needs_hf_models(cfg):
                print(f"SKIP {hid}: requires Hugging Face models (set RUN_DENSE=True)")
                continue
            run_dir = run_experiment(cfg)
            results[hid] = run_dir
            print(f"{hid}: {run_dir.relative_to(PROJECT_ROOT)}")
        except Exception:
            print(f"FAILED: {cfg_path}")
            traceback.print_exc()
            if not continue_on_error:
                raise
    return results


def _hypothesis_order(hid: str) -> tuple:
    match = re.fullmatch(r"H(\d+)", hid or "")
    return (0, int(match.group(1))) if match else (1, hid or "")


def headline_columns() -> list[str]:
    return [f"{name}@{HEADLINE_K}" for name in HEADLINE_METRICS] + \
           [f"article.{name}@{HEADLINE_K}" for name in HEADLINE_METRICS]


def _headline_from_table(table: dict[str, dict[str, dict[str, float]]]) -> dict[str, float | None]:
    out = {}
    for column in headline_columns():
        level, name, k = _split_metric(column)
        out[column] = table.get(level, {}).get(name, {}).get(f"@{k}")
    return out


def _write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(dict.fromkeys(k for row in rows for k in row)))
        writer.writeheader()
        writer.writerows(rows)


def compare_experiments(runs_dir: Path = RUNS_DIR, reports_dir: Path = REPORTS_DIR,
                        latest_only: bool = True) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    '''Writes reports/summary.csv (one row per hypothesis) and reports/vs_baseline.csv
    (headline metric deltas vs the config each hypothesis extends).'''
    rows = []
    for path in Path(runs_dir).glob("*/*/summary.json"):
        s = json.loads(path.read_text(encoding="utf-8"))
        if s.get("status") != "completed":
            continue
        rows.append({
            "hypothesis": s["hypothesis"],
            "baseline": s.get("baseline"),
            "description": s.get("description", ""),
            **_headline_from_table(s.get("metrics", {})),
            "queries": s.get("queries"),
            "records": s.get("records"),
            "query_mean_ms": s.get("timing", {}).get("query_mean_ms"),
            "index_seconds": s.get("timing", {}).get("index_seconds"),
            "peak_gpu_gb": s.get("resources", {}).get("peak_gpu_gb"),
            "run_id": s["run_id"],
            "_changed": "; ".join(s.get("changed", [])),
        })
    if latest_only:
        # run_id starts with a UTC timestamp, so string order = time order.
        latest: dict[str, dict[str, Any]] = {}
        for row in rows:
            if row["hypothesis"] not in latest or row["run_id"] > latest[row["hypothesis"]]["run_id"]:
                latest[row["hypothesis"]] = row
        rows = list(latest.values())
    rows.sort(key=lambda r: (_hypothesis_order(r["hypothesis"]), r["run_id"]))
    if not rows:
        print("No completed runs found.")
        return [], []

    by_hid: dict[str, dict[str, Any]] = {}
    for row in rows:
        by_hid[row["hypothesis"]] = row   # latest wins (rows are sorted by run_id)
    deltas = []
    for row in by_hid.values():
        base = by_hid.get(row["baseline"])
        if base is None:
            continue
        delta: dict[str, Any] = {"hypothesis": row["hypothesis"], "baseline": base["hypothesis"],
                                 "changed": row["_changed"]}
        for column in headline_columns() + ["query_mean_ms"]:
            a, b = row.get(column), base.get(column)
            delta[f"Δ {column}"] = round(a - b, 4) if a is not None and b is not None else None
        deltas.append(delta)

    summary_rows = [{k: v for k, v in row.items() if not k.startswith("_")} for row in rows]
    _write_csv(Path(reports_dir) / "summary.csv", summary_rows)
    if deltas:
        _write_csv(Path(reports_dir) / "vs_baseline.csv", deltas)
    print(f"Wrote {len(summary_rows)} runs to {(Path(reports_dir) / 'summary.csv').relative_to(PROJECT_ROOT)}"
          + (f" and {len(deltas)} comparisons to {(Path(reports_dir) / 'vs_baseline.csv').relative_to(PROJECT_ROOT)}"
             if deltas else ""))
    return summary_rows, deltas


def latest_run(hid: str, runs_dir: Path = RUNS_DIR) -> Path:
    runs = sorted(p.parent for p in Path(runs_dir).glob(f"{hid}/*/summary.json"))
    if not runs:
        raise FileNotFoundError(f"No completed run for {hid} in {runs_dir}")
    return runs[-1]


def compare_queries(hid_a: str, hid_b: str) -> None:
    '''Rank of the first relevant result per query for two hypotheses (latest runs).'''
    a = {r["query_id"]: r for r in read_jsonl(latest_run(hid_a) / "per_query.jsonl")}
    b = {r["query_id"]: r for r in read_jsonl(latest_run(hid_b) / "per_query.jsonl")}

    def rank(row: dict[str, Any] | None) -> str:
        if not row or not row.get("first_hit_rank"):
            return "n/a"
        first = row["first_hit_rank"]
        return f"{first.get('strict') or '-'} / {first.get('article') or '-'}"

    print(f"First relevant rank (strict / article), '-' = not in top-k\n")
    print(f"{'query_id':<16} {hid_a:>9} {hid_b:>9}   query")
    for qid in sorted(a.keys() | b.keys()):
        query = (a.get(qid) or b.get(qid))["query"]
        print(f"{qid:<16} {rank(a.get(qid)):>9} {rank(b.get(qid)):>9}   {query[:70]}")


def show_metrics(run_dir: Path) -> None:
    '''Print summary.json metrics as a small table (rows = level/metric, columns = k).'''
    summary = json.loads((Path(run_dir) / "summary.json").read_text(encoding="utf-8"))
    print(f"{summary['hypothesis']} (baseline {summary['baseline']}): {summary['description']}")
    print(f"changed: {summary['changed']} | queries: {summary.get('queries')} | timing: {summary['timing']}")
    for level, names in summary.get("metrics", {}).items():
        for name, values in names.items():
            print(f"  {level:<8} {name:<10} " + "  ".join(f"{k}={v:.3f}" for k, v in values.items()))


def show_hits(hits: list[SearchHit], width: int = 160) -> None:
    for h in hits:
        text = (h.record.text if h.record else "").replace("\n", " | ")
        print(f"#{h.rank:<2} score={h.score:.4f}  {h.record_id}")
        print(f"     {text[:width]}{'…' if len(text) > width else ''}")

## 2. Sanity checks

Quick unit checks, then load and validate every H1–H11 config.

In [22]:
# metrics (tests/test_metrics.py)
m = metrics_at_k([0, 1, 0, 1], relevant_count=2, k=4)
assert m["hit_rate"] == 1.0 and m["precision"] == 0.5 and m["recall"] == 1.0 and m["mrr"] == 0.5
assert 0.0 < m["ndcg"] <= 1.0
ra, rb = RetrievalRecord("chunk-a", "", "", ["unit-1"]), RetrievalRecord("chunk-b", "", "", ["unit-1"])
_cfg = {"evaluation": {"match_by": "source_unit", "metrics": ["recall", "ndcg", "map"], "k_values": [2]}}
assert evaluate_query([SearchHit("chunk-a", 1.0, record=ra), SearchHit("chunk-b", 0.9, record=rb)], ["unit-1"], _cfg) \
    == {"recall@2": 1.0, "ndcg@2": 1.0, "map@2": 1.0}

# article-level matching: an Article hit covers a gold Point inside it
_art = RetrievalRecord("doc__article-3", "", "", ["doc__article-3"])
_u2a = {"doc__article-3": "doc|3||", "doc__article-3__clause-1__point-a": "doc|3||"}
_cfg = {"evaluation": {"match_by": "source_unit", "also_match_by": ["article"], "metrics": ["recall"], "k_values": [1]}}
assert evaluate_query([SearchHit("doc__article-3", 1.0, record=_art)], ["doc__article-3__clause-1__point-a"], _cfg, _u2a) \
    == {"recall@1": 0.0, "article.recall@1": 1.0}

# fusion (tests/test_fusion.py)
assert rrf_fusion({"sparse": [SearchHit("a", 10, 1), SearchHit("b", 8, 2)],
                   "dense": [SearchHit("b", .9, 1), SearchHit("c", .8, 2)]})[0].record_id == "b"
assert {h.record_id for h in weighted_fusion(
    {"sparse": [SearchHit("a", 10, 1), SearchHit("b", 5, 2)], "dense": [SearchHit("c", .9, 1), SearchHit("b", .1, 2)]},
    {"sparse": .5, "dense": .5})} == {"a", "b", "c"}

# temporal (tests/test_temporal.py)
_tcfg = {"enabled": True, "reference_date": "2026-09-19", "strip_time_from_query": True}
assert parse_temporal_query("hỗ trợ ngư dân", _tcfg).mode == "current"
_p = parse_temporal_query("năm 2022 ngư dân được hỗ trợ thế nào", _tcfg)
assert _p.mode == "as_of" and _p.start == date(2022, 1, 1) and "2022" not in _p.semantic_query
assert parse_temporal_query("Luật Đất đai 2024 quy định gì", _tcfg).mode == "current"
_p = parse_temporal_query("trước năm 2020 có quy định nào", _tcfg)
assert record_matches_temporal({"unit_id": "old", "effective_from": "01/01/2018", "effective_to": "31/12/2019"}, _p, "error")
assert not record_matches_temporal({"unit_id": "new", "effective_from": "01/01/2021", "effective_to": None}, _p, "error")
print("Unit sanity checks: OK")

Unit sanity checks: OK


In [23]:
CONFIG_FILES = {
    "H1": "configs/examples/h1_sparse_hierarchical.yaml",
    "H2": "configs/examples/h2_dense_hierarchical.yaml",
    "H3": "configs/examples/h3_hybrid_rrf.yaml",
    "H4": "configs/examples/h4_hybrid_rerank.yaml",
    "H5": "configs/examples/h5_fixed_length_hybrid.yaml",
    "H6": "configs/examples/h6_fixed_length_sparse.yaml",
    "H7": "configs/examples/h7_child.yaml",
    "H8": "configs/examples/h8_contextual_child.yaml",
    "H9": "configs/examples/h9_parent_to_child.yaml",
    "H10": "configs/examples/h10_temporal_hierarchical.yaml",
    "H11": "configs/examples/h11_parent_to_child_temporal.yaml",
}

for hid, path in CONFIG_FILES.items():
    cfg = load_config(path)
    diff = cfg["_hypothesis_diff"]
    baseline = Path(cfg["_baseline_config_path"]).name if cfg["_baseline_config_path"] else None
    print(f"{hid:<4} mode={cfg['representation']['mode']:<18} method={cfg['retrieval']['method']:<6} "
          f"fusion={cfg['fusion']['method']:<5} rerank={cfg['reranking']['enabled']!s:<5} "
          f"temporal={cfg['temporal']['enabled']!s:<5} dense_model={cfg['indexing']['dense']['model_name'] if cfg['indexing']['dense']['enabled'] else '-'}")
    print(f"     extends {baseline}; actual changes = {diff['actual_changes']}")

H1   mode=hierarchical_units method=sparse fusion=none  rerank=False temporal=False dense_model=-
     extends base.yaml; actual changes = []
H2   mode=hierarchical_units method=dense  fusion=none  rerank=False temporal=False dense_model=BAAI/bge-m3
     extends h1_sparse_hierarchical.yaml; actual changes = ['indexing.dense.enabled', 'indexing.sparse.enabled', 'retrieval.method']
H3   mode=hierarchical_units method=hybrid fusion=rrf   rerank=False temporal=False dense_model=BAAI/bge-m3
     extends h1_sparse_hierarchical.yaml; actual changes = ['fusion.method', 'indexing.dense.enabled', 'retrieval.method']
H4   mode=hierarchical_units method=hybrid fusion=rrf   rerank=True  temporal=False dense_model=BAAI/bge-m3
     extends h3_hybrid_rrf.yaml; actual changes = ['reranking.enabled', 'reranking.models[0]']
H5   mode=fixed_length       method=hybrid fusion=rrf   rerank=False temporal=False dense_model=BAAI/bge-m3
     extends h3_hybrid_rrf.yaml; actual changes = ['ranking.deduplicate_by'

## 3. Data & record checks

In [24]:
_cfg_h1 = load_config(CONFIG_FILES["H1"])
units = load_enriched_units(_cfg_h1)
print("Units (article/clause/point):", len(units))
print("By type:", dict(Counter(u["unit_type"] for u in units)))
print("By document:", dict(Counter(u["doc_id"] for u in units)))
print("Missing effective_from:", sum(1 for u in units if not u.get("effective_from")))

2026-10-19 09:23:41,188 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records


Units (article/clause/point): 9853
By type: {'article': 945, 'clause': 4327, 'point': 4581}
By document: {'101-2024-NĐ-CP': 964, '102-2024-NĐ-CP': 1496, '103-2024-NĐ-CP': 770, '27-2023-QH15': 1587, '29-2023-QH15': 813, '291-2025-NĐ-CP': 181, '31-2024-QH15': 2626, '43-2024-QH15': 11, '54-2026-NĐ-CP': 89, '95-2024-NĐ-CP': 981, '96-2024-NĐ-CP': 335}
Missing effective_from: 0


Duplicate `unit_id` check (raw corpus vs `corpus.duplicate_unit_id_policy`):

In [25]:
_raw_cfg = deepcopy(_cfg_h1)
_raw_cfg["corpus"]["duplicate_unit_id_policy"] = "keep"
_raw = load_enriched_units(_raw_cfg)
_by_id = defaultdict(list)
for u in _raw:
    _by_id[u["unit_id"]].append(u)
dups = {k: v for k, v in _by_id.items() if len(v) > 1}
dups_diff_text = [k for k, v in dups.items() if any(x["text"] != v[0]["text"] for x in v)]
print(f"Raw: {len(_by_id)} unique unit_id / {len(_raw)} rows")
print(f"Duplicated unit_id: {len(dups)} ({sum(len(v) - 1 for v in dups.values())} extra rows); "
      f"with different text: {len(dups_diff_text)}")
print("By document:", dict(Counter(v[0]["doc_id"] for v in dups.values())))
print(f"After policy {_cfg_h1['corpus']['duplicate_unit_id_policy']!r}: {len(units)} rows, "
      f"{sum('_record_id' in u for u in units)} kept as #dupN records")
if dups_diff_text:
    k = dups_diff_text[0]
    print("\nExample with different text:", k)
    for x in dups[k]:
        print("  -", x["text"][:200].replace("\n", " "))

Raw: 9442 unique unit_id / 10996 rows
Duplicated unit_id: 1337 (1554 extra rows); with different text: 206
By document: {'101-2024-NĐ-CP': 492, '102-2024-NĐ-CP': 389, '103-2024-NĐ-CP': 197, '27-2023-QH15': 2, '291-2025-NĐ-CP': 35, '31-2024-QH15': 160, '54-2026-NĐ-CP': 6, '95-2024-NĐ-CP': 45, '96-2024-NĐ-CP': 11}
After policy 'dedupe': 9853 rows, 411 kept as #dupN records

Example with different text: 101-2024-NĐ-CP__article-16__clause-4__point-b__v01/08/2024
  - b) Hồ sơ lưu trữ kết quả bảo vệ, cải tạo, phục hồi đất cấp tỉnh quy định tại điểm d khoản 2 và khoản 3 Điều này được lưu trữ 01 bộ tại Bộ Tài nguyên và Môi trường, 01 bộ tại Bộ Nông nghiệp và Phát tri
  - b) Hồ sơ lưu trữ kết quả bảo vệ, cải tạo, phục hồi đất cấp tỉnh quy định tại điểm d khoản 2 và khoản 3 Điều này được lưu trữ 01 bộ tại Bộ Tài nguyên và Môi trường, 01 bộ tại Bộ Nông nghiệp và Phát tri


In [26]:
for hid in ["H1", "H6", "H7", "H8", "H9"]:
    _cfg = load_config(CONFIG_FILES[hid])
    recs = build_records(_cfg)
    roles = Counter(r.metadata.get("retrieval_role", r.metadata.get("unit_type")) for r in recs)
    lens = [len(r.text.split()) for r in recs]
    print(f"{hid}: {len(recs):>6} records | {dict(roles)} | words/record mean={np.mean(lens):.0f} max={max(lens)}")

print("\nExample H8 record (child + parent heading):")
_r = next(r for r in build_records(load_config(CONFIG_FILES["H8"])) if r.metadata.get("retrieval_role") == "child")
print(_r.record_id)
print(_r.text[:600])

2026-10-19 09:23:41,414 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records
2026-10-19 09:23:42,808 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records


H1:   9853 records | {'article': 945, 'clause': 4327, 'point': 4581} | words/record mean=134 max=1852


2026-10-19 09:23:45,020 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records


H6:   7323 records | {'fixed_chunk': 7323} | words/record mean=220 max=220


2026-10-19 09:23:45,935 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records


H7:   8908 records | {'child': 8908} | words/record mean=75 max=1155


2026-10-19 09:23:47,097 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records


H8:   8908 records | {'child': 8908} | words/record mean=96 max=1196


2026-10-19 09:23:48,273 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records


H9:   9853 records | {'parent': 945, 'child': 8908} | words/record mean=95 max=1390

Example H8 record (child + parent heading):
101-2024-NĐ-CP__article-1__clause-1__v01/08/2024
[PARENT] Điều 1. Phạm vi điều chỉnh
[CITATION] Khoản 1 Điều 1 NGHỊ ĐỊNH Quy định về điều tra cơ bản đất đai; đăng ký, cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu tài sản gắn liền với đất và Hệ thống thông tin đất đai 2024
[CHILD] 1. Nghị định này quy định chi tiết và hướng dẫn thi hành Luật Đất đai về điều tra cơ bản đất đai; đăng ký, cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu tài sản gắn liền với đất và Hệ thống thông tin quốc gia về đất đai, gồm: khoản 4 Điều 50, khoản 1 Điều 55, khoản 5 Điều 131, khoản 7 Điều 135, khoản 10 Điều 138, khoản 6 Điều 139, khoản 7 Điều 140, khoản


In [27]:
# Every relevant_unit_id must exist in the corpus.
benchmark = load_benchmark({"evaluation": {**_cfg_h1["evaluation"], "benchmark_path": BENCHMARK_PATH}})
unit_ids = {u["unit_id"] for u in units}
print(f"{len(benchmark)} queries in {BENCHMARK_PATH}")
for q in benchmark:
    missing = [rid for rid in q.relevant_ids if rid not in unit_ids]
    print(f"- {q.query_id}: {q.query!r} filters={q.filters} | relevant={len(q.relevant_ids)} | missing_in_corpus={missing}")

1 queries in data/benchmarks/schema_example.jsonl
- example-001: 'Nguyên tắc đo đạc lập bản đồ địa chính là gì?' filters={'doc_id': '101-2024-NĐ-CP'} | relevant=1 | missing_in_corpus=[]


## 4. Single query — `scripts/retrieve.py`

Change `QUERY`, `FILTERS` and the config to inspect results.

In [28]:
QUERY = "Nguyên tắc đo đạc lập bản đồ địa chính là gì?"
FILTERS = {}            # e.g. {"doc_id": "101-2024-NĐ-CP"}

cfg = load_notebook_config(CONFIG_FILES["H1"])
pipeline = RetrievalPipeline(cfg)
hits = pipeline.retrieve(QUERY, top_k=5, filters=FILTERS)
print("query_context:", pipeline.last_query_context)
show_hits(hits)

2026-10-19 09:23:49,243 | INFO | legal_retrieval | Building retrieval records...
2026-10-19 09:23:49,330 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records
2026-10-19 09:23:50,642 | INFO | legal_retrieval | Built 9853 retrieval records
2026-10-19 09:23:50,642 | INFO | legal_retrieval | Building BM25 index...
2026-10-19 09:23:51,034 | INFO | legal_retrieval | Index ready in memory (ff5c3c40710b)


query_context: {'temporal': None, 'eligible_record_count': 9853, 'filters': {}}
#1  score=28.2436  101-2024-NĐ-CP__article-3__v01/08/2024
     [CITATION] Điều 3 NGHỊ ĐỊNH Quy định về điều tra cơ bản đất đai; đăng ký, cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu tài sản gắn liền với đất và Hệ thố…
#2  score=28.0869  31-2024-QH15__article-50__clause-1__v01/01/2025
     [CITATION] Khoản 1 Điều 50 Luật Đất đai 2024 | [TITLE] Luật Đất đai > Chương IV. Địa giới đơn vị hành chính, điều tra cơ bản về đất đai > Điều 50. Đo đạc lập bả…
#3  score=27.9225  101-2024-NĐ-CP__article-3__clause-1__v01/08/2024
     [CITATION] Khoản 1 Điều 3 NGHỊ ĐỊNH Quy định về điều tra cơ bản đất đai; đăng ký, cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu tài sản gắn liền với đất v…
#4  score=25.5562  101-2024-NĐ-CP__article-3__clause-1__point-c__v01/08/2024
     [CITATION] Điểm c Khoản 1 Điều 3 NGHỊ ĐỊNH Quy định về điều tra cơ bản đất đai; đăng ký, cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu tài

In [29]:
# Temporal query: H9 (parent -> child) vs H11 (same + temporal filtering)
QUERY_T = "tại thời điểm 01/06/2025 nguyên tắc đo đạc lập bản đồ địa chính"
for hid in ["H9", "H11"]:
    _pipe = RetrievalPipeline(load_notebook_config(CONFIG_FILES[hid]))
    _hits = _pipe.retrieve(QUERY_T, top_k=5)
    print(f"--- {hid} | query_context:", _pipe.last_query_context)
    show_hits(_hits)
    print("Top-1 component_scores:", _hits[0].component_scores if _hits else None, "\n")

2026-10-19 09:23:51,119 | INFO | legal_retrieval | Building retrieval records...
2026-10-19 09:23:51,200 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records
2026-10-19 09:23:52,140 | INFO | legal_retrieval | Built 9853 retrieval records
2026-10-19 09:23:52,140 | INFO | legal_retrieval | Building BM25 index...
2026-10-19 09:23:52,460 | INFO | legal_retrieval | Index ready in memory (9472e4128ed8)


--- H9 | query_context: {'temporal': None, 'eligible_record_count': 9853, 'filters': {}}
#1  score=0.0279  101-2024-NĐ-CP__article-3__clause-1__v01/08/2024
     [PARENT] Điều 3. Nguyên tắc, mục đích đo đạc lập bản đồ địa chính | [CITATION] Khoản 1 Điều 3 NGHỊ ĐỊNH Quy định về điều tra cơ bản đất đai; đăng ký, cấp Giấy c…
#2  score=0.0276  101-2024-NĐ-CP__article-3__clause-1__point-c__v01/08/2024
     [PARENT] Điều 3. Nguyên tắc, mục đích đo đạc lập bản đồ địa chính | [CITATION] Điểm c Khoản 1 Điều 3 NGHỊ ĐỊNH Quy định về điều tra cơ bản đất đai; đăng ký, cấp…
#3  score=0.0273  101-2024-NĐ-CP__article-3__clause-2__v01/08/2024
     [PARENT] Điều 3. Nguyên tắc, mục đích đo đạc lập bản đồ địa chính | [CITATION] Khoản 2 Điều 3 NGHỊ ĐỊNH Quy định về điều tra cơ bản đất đai; đăng ký, cấp Giấy c…
#4  score=0.0271  101-2024-NĐ-CP__article-3__clause-1__point-d__v01/08/2024
     [PARENT] Điều 3. Nguyên tắc, mục đích đo đạc lập bản đồ địa chính | [CITATION] Điểm d Khoản 1 Điều 3 NGHỊ ĐỊNH Quy định

## 5. Single experiment — `scripts/run_experiment.py`

In [ ]:
run_dir = run_experiment(load_notebook_config(CONFIG_FILES["H1"]))
print("Run:", run_dir.relative_to(PROJECT_ROOT))
print("Files:", sorted(p.name for p in run_dir.iterdir()))
show_metrics(run_dir)

In [ ]:
# Inspect per_query.jsonl (✓ = exact gold unit, ≈ = same Article only)
for row in read_jsonl(run_dir / "per_query.jsonl"):
    print(row["query_id"], "|", row["query"])
    print("  relevant:", row["relevant_unit_ids"])
    print("  first relevant rank:", row["first_hit_rank"], "| metrics:", row["metrics"])
    for r in row["results"][:5]:
        mark = "✓" if r["hit"] else "≈" if r.get("article_hit") else " "
        print(f"  {mark} #{r['rank']:<2} {r['score']:.3f} {r['record_id']}")
        print(f"         {r['snippet']}")

## 6. Full suite H1–H11 — `scripts/run_suite.py`

With `RUN_DENSE = False`, H2–H5 are skipped. Other suites: `configs/suites/example_suite.yaml` (H1–H3), `configs/suites/parent_child_temporal_suite.yaml` (BM25 only: H1, H7–H11).

In [32]:
suite_results = run_suite("configs/suites/all_hypotheses.yaml", skip_hf_models=not RUN_DENSE)

2026-10-19 09:23:52,838 | INFO | legal_retrieval | Hypothesis baseline: /home/barokhuu/workspace/URA/notebook/configs/base.yaml
2026-10-19 09:23:52,838 | INFO | legal_retrieval | Declared changes = []
2026-10-19 09:23:52,839 | INFO | legal_retrieval | Actual changes = []
2026-10-19 09:23:52,943 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records


H1: artifacts/runs/H1/20261019T022352.833410Z-ce8ae139


2026-10-19 09:23:53,157 | INFO | legal_retrieval | Hypothesis baseline: /home/barokhuu/workspace/URA/notebook/configs/examples/h1_sparse_hierarchical.yaml
2026-10-19 09:23:53,157 | INFO | legal_retrieval | Declared changes = ['ranking.deduplicate_by', 'representation.fixed_length.chunk_size', 'representation.fixed_length.overlap', 'representation.fixed_length.source_unit_types', 'representation.fixed_length.unit', 'representation.mode']
2026-10-19 09:23:53,158 | INFO | legal_retrieval | Actual changes = ['ranking.deduplicate_by', 'representation.mode']


SKIP H2: requires Hugging Face models (set RUN_DENSE=True)
SKIP H3: requires Hugging Face models (set RUN_DENSE=True)
SKIP H4: requires Hugging Face models (set RUN_DENSE=True)
SKIP H5: requires Hugging Face models (set RUN_DENSE=True)


2026-10-19 09:23:53,177 | INFO | legal_retrieval | Building retrieval records...
2026-10-19 09:23:53,265 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records
2026-10-19 09:23:55,347 | INFO | legal_retrieval | Built 7323 retrieval records
2026-10-19 09:23:55,348 | INFO | legal_retrieval | Building BM25 index...
2026-10-19 09:23:55,797 | INFO | legal_retrieval | Index ready in memory (772709361c0b)
2026-10-19 09:23:55,885 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records
2026-10-19 09:23:55,940 | INFO | legal_retrieval | Hypothesis baseline: /home/barokhuu/workspace/URA/notebook/configs/examples/h1_sparse_hierarchical.yaml
2026-10-19 09:23:55,941 | INFO | legal_retrieval | Declared changes = ['representation.mode']
2026-10-19 09:23:55,941 | INFO | legal_retrieval | Actual changes = ['representation.mode']
2026-10-19 09:23:55,960 | INFO | legal_retrieval | Build

...

H7: artifacts/runs/H7/20261019T022355.936074Z-8ee672f7


2026-10-19 09:23:58,139 | INFO | legal_retrieval | Built 8908 retrieval records
2026-10-19 09:23:58,140 | INFO | legal_retrieval | Building BM25 index...
2026-10-19 09:23:58,390 | INFO | legal_retrieval | Index ready in memory (92801c631f46)
2026-10-19 09:23:58,479 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records
2026-10-19 09:23:58,548 | INFO | legal_retrieval | Hypothesis baseline: /home/barokhuu/workspace/URA/notebook/configs/examples/h8_contextual_child.yaml
2026-10-19 09:23:58,549 | INFO | legal_retrieval | Declared changes = ['representation.parent_child.include_parent_records', 'retrieval.parent_child.enabled']
2026-10-19 09:23:58,550 | INFO | legal_retrieval | Actual changes = ['representation.parent_child.include_parent_records', 'retrieval.parent_child.enabled']
2026-10-19 09:23:58,653 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records


H8: artifacts/runs/H8/20261019T022357.161149Z-ae45d06d
H9: artifacts/runs/H9/20261019T022358.540439Z-4bc9cfae


2026-10-19 09:23:58,708 | INFO | legal_retrieval | Hypothesis baseline: /home/barokhuu/workspace/URA/notebook/configs/examples/h1_sparse_hierarchical.yaml
2026-10-19 09:23:58,709 | INFO | legal_retrieval | Declared changes = ['temporal.enabled']
2026-10-19 09:23:58,709 | INFO | legal_retrieval | Actual changes = ['temporal.enabled']
2026-10-19 09:23:58,806 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records
2026-10-19 09:23:58,874 | INFO | legal_retrieval | Hypothesis baseline: /home/barokhuu/workspace/URA/notebook/configs/examples/h9_parent_to_child.yaml
2026-10-19 09:23:58,875 | INFO | legal_retrieval | Declared changes = ['temporal.enabled']
2026-10-19 09:23:58,875 | INFO | legal_retrieval | Actual changes = ['temporal.enabled']
2026-10-19 09:23:58,971 | INFO | legal_retrieval | Duplicate unit_id: dropped 1143 identical rows, kept 411 differing rows as #dupN records


H10: artifacts/runs/H10/20261019T022358.703923Z-5f523a43
H11: artifacts/runs/H11/20261019T022358.869050Z-dc3dc2c6


## 7. Compare runs — `scripts/compare_experiments.py`

Writes `artifacts/reports/summary.csv` (latest run of each hypothesis) and `artifacts/reports/vs_baseline.csv`. Set `latest_only=False` to list every run in `summary.csv`.

In [ ]:
summary_rows, delta_rows = compare_experiments(latest_only=True)
try:
    import pandas as pd
    display(pd.DataFrame(summary_rows).drop(columns=["description", "run_id"]))  # noqa: F821  (Jupyter built-in)
except (ImportError, NameError):
    for r in summary_rows:
        print(r["hypothesis"], {c: r[c] for c in headline_columns()})

Each hypothesis against its own baseline (the config it `extends`). Positive Δ = better than the baseline (for `query_mean_ms`, positive = slower).

In [ ]:
try:
    display(pd.DataFrame(delta_rows).drop(columns=["changed"]))  # noqa: F821
except (ImportError, NameError):
    for d in delta_rows:
        print(d["hypothesis"], "vs", d["baseline"], {k: v for k, v in d.items() if k.startswith("Δ")})

In [ ]:
# Per query: rank of the first relevant result in two hypotheses (change the IDs)
compare_queries("H3", "H4")

In [35]:
# Free memory (cached indexes and models)
clear_index_cache()